# Inkling-2-Mini Uncensored (AtomicChat) — Colab chat v1: **Q4_K** • vision • native max reasoning • tool calls • web search • 4K default context • public link

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (preselected if you opened this file in Colab). An L4 / A100 works too and is faster.
2. **Cell 1** — pick the model file (default **Q4_K**) and run it. It mounts Google Drive first (optional build cache), then builds llama.cpp **b12620** and downloads the pre-quantized GGUF **plus the vision encoder** (`mmproj`, 0.64 GB) *in parallel* (~25 min the first time because the file is 16.5 GB; re-running is fast). No conversion needed.
3. **Cell 2** — keep the defaults (**4,096-token** context, **Max** reasoning, **Parallel** web search, **vision on**) or change them, then run it. It shows the chat right in the notebook, plus a private link and a **public link**.
4. **Keep this tab open and leave Cell 2 running.** It watches the model server and the link and repairs them automatically.

**What makes this model different (and how the notebook handles it)**
- **A looped mixture-of-experts.** Inkling-2-Mini (Thinking Machines Lab) has 26.4B parameters, 3.4B active per token, 128 experts per layer (8 used). It stores 24 layers, but layers 6–17 run **three times** with shared weights, so every token makes 48 layer visits. This needs llama.cpp **b12600 or newer** (the `inkling2_loop` architecture); the notebook pins **b12620** and stops early, before the 15-minute build, if the checkout does not contain the architecture.
- **Q4_K is bigger than a free T4.** The 16.5 GB file does not fit in 15 GB of GPU memory, so Cell 2 reads the real size of every layer's experts from the GGUF and puts the experts of just enough **outer** layers (23→18, then 5→0) in system RAM — about 7 of them at the default 4K window; the attention, the router and the whole looped core stay on the GPU. It is slower than a file that fits entirely; **IQ3_S** needs only ~2 layers in RAM and **IQ2_M** fits a T4 completely (lowest quality). On an L4 / A100 nothing goes to RAM. If a load runs out of memory, Cell 2 moves two more layers to RAM and retries by itself.
- **Why never the looped layers:** as of llama.cpp b12600–b12620, experts of layers 6–17 in system RAM produce garbage output (ggml-org/llama.cpp #32710, #32755), and when any experts are in RAM the GPU "op-offload" must be off (`--no-op-offload`) or long prompts get corrupted (#32778). The notebook always does both. Cell 2 also *tests a ~300-token prompt at start-up* and, if it comes back garbled, restarts itself in a slower safe mode.
- **Native reasoning, always on, maximum by default.** The chat template takes a `reasoning_effort` and writes “Thinking effort level: …” into the prompt; Cell 2 passes **`max`** (0.99) with no thinking-token limit. The reasoning appears in the collapsible panel. Other levels: High 0.9, Medium 0.7, Low 0.2, Minimal 0.1, None 0 — and the model card notes that even *None* still thinks briefly. No prompt tricks are used. Sampling is the vendor's: temperature 1.0, no top-k / min-p.
- **Reasoning loops.** The model card warns that this small model sometimes re-checks its work endlessly. Replies are therefore capped at **32,768 tokens** (`MAX_TOKENS` in the advanced block, `-1` = unlimited) and the chat UI stops after 100 agent steps.
- **Context: 4,096 tokens by default, up to 1,048,576 on a big GPU.** Only the 8 global-attention visits (of 48) keep a long-range chat memory (the other 40 use a 1,024-token window), so the KV cache is small: roughly 24 KiB per token at 16-bit. **But llama.cpp runs this model's attention without flash-attention, so its working buffers hold a score matrix of context × batch × 24 heads** — measured at 25.5 GB for a 262K window at the default batch of 512 (that is what makes a naive “Max” run out of memory). Cell 2 therefore models those buffers and **shrinks the prompt batch (`-ub`, 512 → 16) as the window grows**: big windows work, but long prompts and files are read more slowly. On a free T4 the practical range is roughly 32K–128K with comfortable speed, and “Max” goes much further (slowly, with a tiny batch and at most 2 extra layers in RAM).
- **4K is tight for Max reasoning** — thinking alone can use thousands of tokens, and this model cannot shift its context, so a chat that fills the window stops with “context full”. For real work pick **16384 or more** in Cell 2. “Max” picks the biggest window that fits.
- **Vision:** attach images with the **+** button. Cell 2 tests the encoder at start-up with a red square, and if that fails it moves the encoder to the CPU or switches vision off, and tells you. Big images use many tokens — at 4K context raise the window or set `IMAGE_MAX_TOKENS`. Audio and video are not supported.
- **Tools & web:** the model gets `web_search` and `web_fetch` (Parallel + Exa + DuckDuckGo with auto-fallback; the Parallel key also adds the slow `web_research` tool — the keys are pre-filled). Results are capped relative to the context size and every call is time-bounded. The first search asks permission — click **Always allow** — or use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button.
- **No context shifting:** if a chat truly fills the window it stops with a “context full” message — start a new chat.
- **About the model file:** AtomicChat's build of Inkling-2-Mini with refusals removed by a rank-1 edit baked into the weights (Apache-2.0). Their own measurements: MMLU 79.2% vs 80.7% for the original, and some answers still open with a caution line. Q4_K: KLD 0.0115 / top-1 95.6% against the BF16 original; IQ3_S 0.0235 / 93.8%; IQ2_M 0.132 / 85.7%. Use responsibly and only where that is allowed.
- **Public link:** a **Cloudflare** link (no account, new address each run). The chat inside the notebook always works too.
- Chats are stored in your browser tab. If the public link changes, it's a new origin, so export chats you care about from the UI settings.


In [ ]:
#@title 1️⃣ Setup — build llama.cpp and download Inkling-2-Mini (Q4_K + vision encoder, ≈25 min first run) { display-mode: "form" }
MODEL_QUANT = "Q4_K · 16.5 GB · best quality (default)" #@param ["Q4_K · 16.5 GB · best quality (default)", "IQ3_S · 13.3 GB · balanced", "IQ2_M · 8.7 GB · smallest, fits a T4 entirely"]
DOWNLOAD_VISION = True #@param {type:"boolean"}
USE_DRIVE_CACHE = True #@param {type:"boolean"}
# MODEL_QUANT: the repo ships pre-quantized GGUFs (the uncensoring edit is baked in) — this is a plain download, no conversion.
#   Inkling-2-Mini is a 26.4B mixture-of-experts model (3.4B active). Q4_K is the best file in the repo, but at 16.5 GB it is
#   bigger than a free 15 GB T4: Cell 2 then keeps the attention + the looped core on the GPU and puts the experts of a few
#   outer layers in system RAM (automatic, a bit slower). IQ3_S needs only ~2 layers in RAM; IQ2_M fits a T4 entirely but is
#   the lowest quality (KLD 0.13, top-1 86% vs 0.012 / 96% for Q4_K). On an L4 / A100 everything fits on the GPU.
# DOWNLOAD_VISION: also downloads the 0.64 GB vision encoder (mmproj file) so you can attach images in the chat.
#   If the chosen quant is missing from the repo, the next-best one is picked automatically.
# USE_DRIVE_CACHE: mounts Google Drive first and saves the compiled server there so the next session skips the ~15 min build.

import os, sys, time, json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

SCRATCH = "/content"
QUANT     = MODEL_QUANT.split()[0]                     # e.g. "Q4_K"
REPO_ID   = "AtomicChat/Inkling-2-Mini-Uncensored-GGUF"
FILENAME  = f"Inkling-2-Mini-Uncensored-AD-{QUANT}.gguf"
MMPROJ_FILE = "mmproj-Inkling-2-Mini-Uncensored-F16.gguf"      # the vision encoder (F16: a T4 has no native BF16)
PREFERENCE = ["Q4_K", "IQ3_S", "IQ2_M"]                # auto-fallback order if the chosen quant is missing
SIZE_GB   = {"Q4_K": 16.53, "IQ3_S": 13.34, "IQ2_M": 8.67}   # for the progress display only
LLAMA_TAG = "b12620"                                   # pinned llama.cpp release (exists; verified). Inkling 2 needs b12600 or newer.
WITH_QUANTIZE = False

LLAMA_DIR = "/content/llama.cpp"
BIN_PATH  = f"{LLAMA_DIR}/build/bin/llama-server"
QBIN_PATH = f"{LLAMA_DIR}/build/bin/llama-quantize"
STAMP     = f"{LLAMA_DIR}/build/inkling2_build_stamp.txt"
MODEL_DIR = "/content/models"
CF_PATH   = "/content/cloudflared"
CONFIG    = "/content/inkling2_mini_config.json"
BUILD_LOG = "/content/build.log"
CF_LOG    = "/content/cf.log"
UI_TGZ    = "/content/llama-ui.tar.gz"
WEBUI_SRC = "/content/webui_src"
CACHE_DIR = "/content/drive/MyDrive/inkling2_cache"
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(MODEL_DIR, exist_ok=True)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, json, struct

# ---------------------------------------------------------------- minimal GGUF header reader
_SCALARS = {0: "<B", 1: "<b", 2: "<H", 3: "<h", 4: "<I", 5: "<i", 6: "<f", 7: "<?", 10: "<Q", 11: "<q", 12: "<d"}
_KEEP_SUFFIX = (".block_count", ".context_length", ".head_count_kv", ".head_count", ".embedding_length", ".key_length",
                ".loop.start", ".loop.end", ".loop.count", ".loop_start", ".loop_end", ".loop_count",
                ".core_start", ".core_end", ".num_passes", ".sliding_window", ".expert_count", ".expert_used_count")

def read_gguf_info(path, want_prefixes=("general.", "tokenizer.chat_template"), buf_mb=192):
    """-> (meta, tensors, data_start).
    meta    : {key: value} for the small keys (scalars + strings); big arrays are skipped.
    tensors : [(name, offset_in_data_section)] for every tensor.
    Pure python, no dependency. Raises ValueError if the file is not a GGUF."""
    with open(path, "rb") as f:
        buf = f.read(buf_mb << 20)
    if buf[:4] != b"GGUF":
        raise ValueError("not a GGUF file")
    ver = struct.unpack_from("<I", buf, 4)[0]
    if ver < 2:
        raise ValueError(f"unsupported GGUF version {ver}")
    n_tensors, n_kv = struct.unpack_from("<QQ", buf, 8)
    pos, out = 24, {}

    def rstr(p):
        n = struct.unpack_from("<Q", buf, p)[0]
        return buf[p + 8:p + 8 + n].decode("utf-8", "replace"), p + 8 + n

    def skip(t, p):
        if t in _SCALARS:
            return p + struct.calcsize(_SCALARS[t])
        if t == 8:
            n = struct.unpack_from("<Q", buf, p)[0]
            return p + 8 + n
        if t == 9:
            et, cnt = struct.unpack_from("<IQ", buf, p)
            p += 12
            if et in _SCALARS:
                return p + cnt * struct.calcsize(_SCALARS[et])
            for _ in range(cnt):
                p = skip(et, p)
            return p
        raise ValueError(f"bad GGUF value type {t}")

    for _ in range(n_kv):
        key, pos = rstr(pos)
        t = struct.unpack_from("<I", buf, pos)[0]
        pos += 4
        keep = key.startswith(want_prefixes) or key.endswith(_KEEP_SUFFIX)
        if keep and t in _SCALARS:
            out[key] = struct.unpack_from(_SCALARS[t], buf, pos)[0]
            pos += struct.calcsize(_SCALARS[t])
        elif keep and t == 8:
            out[key], pos = rstr(pos)
        else:
            pos = skip(t, pos)
    tensors = []
    for _ in range(n_tensors):
        name, pos = rstr(pos)
        n_dims = struct.unpack_from("<I", buf, pos)[0]
        pos += 4 + 8 * n_dims + 4                       # n_dims, the dims, the ggml type
        off = struct.unpack_from("<Q", buf, pos)[0]
        pos += 8
        tensors.append((name, off))
    align = out.get("general.alignment") or 32
    align = int(align) if isinstance(align, (int, float)) and align > 0 else 32
    return out, tensors, (pos + align - 1) // align * align

def read_gguf_meta(path, **kw):
    return read_gguf_info(path, **kw)[0]

def expert_bytes_by_layer(tensors, data_start, file_size):
    """{layer: bytes of that layer's routed-expert tensors} (blk.N.ffn_*_exps*), sizes taken from the offset gaps."""
    order = sorted(tensors, key=lambda t: t[1])
    out = {}
    for i, (name, off) in enumerate(order):
        end = order[i + 1][1] if i + 1 < len(order) else file_size - data_start
        m = re.match(r"blk\.(\d+)\.ffn_.*_exps", name)
        if m:
            out[int(m.group(1))] = out.get(int(m.group(1)), 0) + max(0, end - off)
    return out


# ---------------------------------------------------------------- the looped architecture
def arch_facts(meta, profile):
    """Layer layout from the GGUF (falls back to the profile): n_layer, loop_start, loop_end, loop_count,
    kv_heads, head_dim, ctx_train."""
    arch = meta.get("general.architecture", "") or ""

    def pos_num(*keys):
        for k in keys:
            v = meta.get(f"{arch}.{k}")
            if isinstance(v, (int, float)) and not isinstance(v, bool) and v > 0:
                return int(v)
        return 0

    def any_num(suffixes):
        for k, v in meta.items():
            if k.endswith(suffixes) and isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0:
                return int(v)
        return None

    n_layer = pos_num("block_count") or profile["N_LAYER"]
    ls = any_num((".loop.start", ".loop_start", ".core_start"))
    le = any_num((".loop.end", ".loop_end", ".core_end"))
    lc = any_num((".loop.count", ".loop_count", ".num_passes"))
    ls = profile["LOOP_START"] if ls is None else ls
    le = profile["LOOP_END"] if le is None else le
    lc = profile["LOOP_COUNT"] if not lc else lc
    if not (0 <= ls < le <= n_layer):
        ls, le = profile["LOOP_START"], profile["LOOP_END"]
    kvh = pos_num("attention.head_count_kv") or profile["N_KV_HEAD"]
    hd = pos_num("attention.key_length")
    hd = hd if 32 <= hd <= 512 else profile["HEAD_DIM"]
    return n_layer, ls, le, lc, kvh, hd, pos_num("context_length")

def full_attn_visits(n_layer, loop_start, loop_end, loop_count, period=6):
    """Layer visits per token that keep a full-length KV cache: every 6th layer is global attention, and the layers
    of the loop are visited `loop_count` times, each visit with its own cache (the other visits use a 1,024-token window)."""
    n = 0
    for i in range(n_layer):
        if (i + 1) % period == 0:
            n += loop_count if loop_start <= i < loop_end else 1
    return n

def host_eligible_layers(n_layer, loop_start, loop_end):
    """Layers whose experts may live in system RAM, in the order they are moved there. The looped layers
    (loop_start..loop_end-1) are NEVER eligible: with llama.cpp b126xx their host-resident experts give garbage output."""
    return list(range(n_layer - 1, loop_end - 1, -1)) + list(range(loop_start - 1, -1, -1))

def host_expert_regex(layers):
    """llama-server --override-tensor argument that puts the routed experts of `layers` in system RAM."""
    return r"blk\.(" + "|".join(str(i) for i in sorted(layers)) + r")\.ffn_.*_exps=CPU"


# ---------------------------------------------------------------- GPU memory planner
KV_BYTES_PER_ELEM = {"f16": 2.0, "q8_0": 34 / 32, "q4_0": 18 / 32}   # ggml block sizes

def kv_mib(ctx, kv, n_layer, n_kv_head, head_dim):
    return 2 * n_layer * n_kv_head * head_dim * KV_BYTES_PER_ELEM[kv] * ctx / 1048576

# The attention of this model is not run through flash-attention in llama.cpp, so the working buffers hold a score matrix
# of (context x prompt batch x 24 heads) floats: measured 25,535 MiB at 262,144 tokens with the default batch of 512.
# So the prompt batch (-ub) has to shrink as the window grows (slower prompt reading, but the window fits).
UB_LADDER = (512, 256, 128, 64, 32, 16)

def compute_mib(ctx, ub):
    """GPU working buffers: ~1.5 MiB per batch token + ~200 bytes per (context token x batch token)."""
    return 1.5 * ub + ctx * ub * 200 / 1048576

def pick_ubatch(ctx, cap_mib=1200):
    """Largest prompt batch whose working buffers stay under cap_mib at this context size."""
    for ub in UB_LADDER:
        if compute_mib(ctx, ub) <= cap_mib:
            return ub
    return UB_LADDER[-1]

def plan_candidates(model_mib, total_mib, ladder, target, kv_pref, dims, overhead_mib, min_free_mib,
                    auto_order=None, host_mib=(), force_k=-1, ub_user=0, max_extra_k=None):
    """Ordered (ctx, kv, k, ub) launch attempts; k = number of layers whose experts go to system RAM, ub = prompt batch.
    Largest context first; at equal context the fewest host layers first, then the best KV precision.
    Entries predicted to fit come first; the rest (a safety net if the estimate is too pessimistic) follow.
    host_mib[i] = GPU MiB freed by moving the i-th eligible layer's experts to RAM. force_k >= 0 pins k, ub_user > 0 pins ub.
    max_extra_k (used for "Max"): never use more than that many host layers beyond what the weights alone need."""
    kv_order = {"auto": auto_order or ["q8_0", "q4_0"], "f16": ["f16"], "q8_0": ["q8_0"], "q4_0": ["q4_0"]}[kv_pref]
    sizes = [c for c in ladder if c <= target]
    kmax = len(host_mib)
    fit, nofit = [], []
    k_base = 0
    if max_extra_k is not None and sizes:                  # host layers needed for the weights at the smallest window
        c0 = sizes[-1]
        need0 = (model_mib + kv_mib(c0, kv_order[-1], *dims) + compute_mib(c0, ub_user if ub_user > 0 else pick_ubatch(c0))
                 + overhead_mib + min_free_mib)
        freed0 = 0.0
        while need0 - freed0 > total_mib and k_base < kmax:
            freed0 += host_mib[k_base]
            k_base += 1
    for c in sizes:
        ub = ub_user if ub_user > 0 else pick_ubatch(c)
        for kv in kv_order:
            need = model_mib + kv_mib(c, kv, *dims) + compute_mib(c, ub) + overhead_mib + min_free_mib
            k, freed = 0, 0.0
            if force_k >= 0:
                k = min(force_k, kmax)
                freed = sum(host_mib[:k])
            else:
                while need - freed > total_mib and k < kmax:
                    freed += host_mib[k]
                    k += 1
            ok = need - freed <= total_mib and (max_extra_k is None or k <= k_base + max_extra_k)
            ub_c = ub
            if ok and ub_user <= 0:                     # spare GPU memory -> a bigger prompt batch (faster reading of long chats)
                slack = total_mib - (need - freed)
                for cand in UB_LADDER:
                    if cand <= ub:
                        break
                    if compute_mib(c, cand) - compute_mib(c, ub) <= 0.8 * slack:
                        ub_c = cand
                        break
            (fit if ok else nofit).append((c, kv, k, ub_c))
    fit.sort(key=lambda t: (-t[0], t[2], kv_order.index(t[1])))
    kk = min(force_k, kmax) if force_k >= 0 else kmax
    return fit + [(c, kv, kk, ub) for (c, kv, _k, ub) in nofit if kv == kv_order[-1]][:3]

def tool_cap_chars(ctx):
    """Max characters one web-tool result may add to the conversation (~1/8 of the window, 3 chars/token)."""
    return int(min(30000, max(2500, ctx * 0.375)))


# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- choose a GGUF file in a repo listing
def pick_gguf(files, quant, preference, exact_name=None):
    """-> (filename, quant_used) or None. Exact name first, then the requested quant, then the next best.
    Vision encoders (mmproj) and LoRA adapters are never picked as the model."""
    ggufs = sorted(f for f in files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                   and "lora" not in f.lower() and "/" not in f and not re.search(r"-\d{5}-of-\d{5}", f))
    if exact_name and exact_name in ggufs:
        return exact_name, quant
    order = preference[preference.index(quant):] if quant in preference else list(preference)
    for q in order:
        pat = re.compile(r"(?<![A-Za-z0-9])" + re.escape(q) + r"(?![A-Za-z0-9])", re.I)
        hit = [f for f in ggufs if pat.search(f)]
        if hit:
            return hit[0], q
    return None


# ---------------------------------------------------------------- tiny PNG for the vision self-test
def make_png_solid(w, h, rgb):
    """A valid w x h solid-colour PNG (no imaging library needed)."""
    import zlib
    def chunk(tag, data):
        c = struct.pack(">I", len(data)) + tag + data
        return c + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
    row = b"\x00" + bytes(rgb) * w
    raw = row * h
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def pick_mmproj(files, preferred=""):
    """Vision-encoder GGUF in a repo listing: the preferred name, else F16 > BF16 > F32 > anything called mmproj*
    (F16 first: a T4 has no native BF16)."""
    cands = sorted(f for f in files if f.lower().startswith("mmproj") and f.lower().endswith(".gguf") and "/" not in f)
    if preferred and preferred in cands:
        return preferred
    for pat in (r"(?<![a-z])f16", r"bf16", r"f32"):
        for f in cands:
            if re.search(pat, f.lower()):
                return f
    return cands[0] if cands else ""
# ==== PURE-END ====


# ======================= shared helpers (identical in both notebooks) =======================
import threading
SRC_LOCK = threading.Lock()
BUILD_STATE = {"done": threading.Event(), "ok": False, "err": ""}

def run(cmd, log, env=None, retries=1):
    for attempt in range(retries):
        with open(log, "a") as f:
            f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        if p.returncode == 0:
            return
        if attempt + 1 < retries:
            time.sleep(5)
    tail = open(log, errors="replace").read()[-4000:]
    raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{tail}")

def binary_ok(path):
    try:
        out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60)
        return out.returncode == 0
    except Exception:
        return False

def find_libcuda():
    """Path of the CUDA driver library (libcuda.so / libcuda.so.1), or '' if absent."""
    paths = []
    try:
        out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout or ""
        for line in out.splitlines():
            if "libcuda.so" in line and "=>" in line:
                paths.append(line.rsplit("=> ", 1)[-1].strip())
    except Exception:
        pass
    paths += ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib64/libcuda.so.1",
              "/usr/local/cuda/compat/libcuda.so.1",
              "/usr/local/cuda/lib64/stubs/libcuda.so",
              "/usr/local/cuda/targets/x86_64-linux/lib/stubs/libcuda.so"]
    for p in paths:
        if p and os.path.exists(p):
            return p
    return ""

def ensure_cuda_driver_link():
    """CMake's FindCUDAToolkit sometimes cannot see the driver library (CUDA::cuda_driver); a libcuda.so
    symlink next to the toolkit libs fixes that. No-op when CMake can already find it."""
    need = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib")
            if os.path.isdir(d) and not os.path.exists(f"{d}/libcuda.so")]
    if not need:
        return ""
    drv = find_libcuda()
    if not drv:
        return ""
    for d in need:
        try:
            os.symlink(drv, f"{d}/libcuda.so")
            print(f"🔧 linked {drv} → {d}/libcuda.so (CUDA::cuda_driver fix)")
        except OSError:
            pass
    return drv

def _clone_llama_source():
    """Pinned llama.cpp checkout (needed by the compiler and, for Cyber-Quack, by convert_hf_to_gguf.py).
    Thread-safe and idempotent: never deletes a checkout that is already in use."""
    with SRC_LOCK:
        marker = f"{LLAMA_DIR}/.pinned_tag"
        ok = (os.path.exists(f"{LLAMA_DIR}/CMakeLists.txt") and os.path.exists(marker)
              and open(marker).read().strip() == LLAMA_TAG)
        if ok:
            return "already there"
        shutil.rmtree(LLAMA_DIR, ignore_errors=True)
        open(BUILD_LOG, "a").close()
        run(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG,
             "https://github.com/ggml-org/llama.cpp", LLAMA_DIR], BUILD_LOG, retries=3)
        open(marker, "w").write(LLAMA_TAG)
        return "cloned"

def ensure_llama_source():
    """Pinned llama.cpp checkout, and a check (before the long build) that it knows the Inkling 2 loop architecture."""
    status = _clone_llama_source()
    hit = subprocess.run(["grep", "-rIl", "inkling2_loop", f"{LLAMA_DIR}/src"], capture_output=True, text=True)
    if hit.returncode != 0 or not hit.stdout.strip():
        shutil.rmtree(LLAMA_DIR, ignore_errors=True)
        raise RuntimeError(f"llama.cpp {LLAMA_TAG} has no Inkling-2-Mini support (architecture inkling2_loop, needs b12600 or newer). "
                           "Set LLAMA_TAG in Cell 1 to a newer release and run it again.")
    return status

def prepare_webui():
    """Download + unpack the official prebuilt chat UI of this exact release (no Node.js needed).
    Cell 2 patches a copy of it and serves it with --path, so the server build never depends on it."""
    if os.path.isfile(f"{WEBUI_SRC}/index.html"):
        return "already there"
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/llama-{LLAMA_TAG}-ui.tar.gz"
    run(["wget", "-q", "--tries=3", "-O", UI_TGZ, url], BUILD_LOG)
    extract_ui(UI_TGZ, WEBUI_SRC)
    return "downloaded"

def build_server():
    want = f"{LLAMA_TAG} sm{ARCH}"
    def good():
        s = os.path.exists(BIN_PATH) and binary_ok(BIN_PATH)
        q = (not WITH_QUANTIZE) or (os.path.exists(QBIN_PATH) and binary_ok(QBIN_PATH))
        return s and q
    if good() and os.path.exists(STAMP) and open(STAMP).read().strip() == want:
        return "already built"
    if USE_CACHE:
        os.makedirs(os.path.dirname(BIN_PATH), exist_ok=True)
        if os.path.exists(CACHED_BIN):
            shutil.copy(CACHED_BIN, BIN_PATH); os.chmod(BIN_PATH, 0o755)
        if WITH_QUANTIZE and os.path.exists(CACHED_QUANT):
            shutil.copy(CACHED_QUANT, QBIN_PATH); os.chmod(QBIN_PATH, 0o755)
        if good():
            open(STAMP, "w").write(want)
            return "restored from Drive cache"
    ensure_llama_source()
    env = dict(os.environ)
    drv = ensure_cuda_driver_link()
    # -DLLAMA_BUILD_UI=OFF: the chat UI is served from files (--path), so a UI download/embedding problem
    # can never break the compile.
    cmake_cmd = ["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build",
                 "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={ARCH}",
                 "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_UI=OFF",
                 "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                 "-DCMAKE_BUILD_TYPE=Release"]
    try:
        run(cmake_cmd, BUILD_LOG, env=env)
    except RuntimeError as e:
        if not (drv and "cuda_driver" in str(e)):
            raise
        run(cmake_cmd + [f"-DCUDA_DRIVER_LIBRARY={drv}"], BUILD_LOG, env=env)
    targets = ["--target", "llama-server"] + (["--target", "llama-quantize"] if WITH_QUANTIZE else [])
    run(["cmake", "--build", f"{LLAMA_DIR}/build", "--config", "Release"] + targets +
        ["-j", str(min(os.cpu_count() or 2, 4))], BUILD_LOG, env=env)
    if not binary_ok(BIN_PATH):
        raise RuntimeError(f"The server was built but does not start. See {BUILD_LOG}")
    if WITH_QUANTIZE and not binary_ok(QBIN_PATH):
        raise RuntimeError(f"llama-quantize was not built. See {BUILD_LOG}")
    open(STAMP, "w").write(want)
    if USE_CACHE:
        try:
            shutil.copy(BIN_PATH, CACHED_BIN)
            if WITH_QUANTIZE:
                shutil.copy(QBIN_PATH, CACHED_QUANT)
        except OSError as e:
            print(f"⚠️ could not save the build to Drive ({e}) - continuing")
    return "built from source"

def build_job():
    """build_server() + signal for the threads that need the binaries (quantize stage)."""
    try:
        res = build_server()
        BUILD_STATE["ok"] = True
        return res
    except BaseException as e:      # noqa: BLE001
        BUILD_STATE["err"] = str(e)[:500]
        raise
    finally:
        BUILD_STATE["done"].set()

def get_cloudflared():
    if os.path.exists(CF_PATH) and os.path.getsize(CF_PATH) > 1_000_000 and binary_ok(CF_PATH):
        return "already there"
    run(["wget", "-q", "--tries=3", "-O", CF_PATH,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        CF_LOG, retries=3)
    os.chmod(CF_PATH, 0o755)
    if os.path.getsize(CF_PATH) < 1_000_000 or not binary_ok(CF_PATH):
        os.remove(CF_PATH)
        raise RuntimeError("cloudflared download looks broken; run this cell again.")
    return "downloaded"

PIP_LOCK = threading.Lock()      # two pip processes at once can corrupt each other

def pip_install(pkgs):
    with PIP_LOCK:
        p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"] + pkgs,
                           capture_output=True, text=True)
    return p.returncode == 0, (p.stderr or p.stdout or "").strip()[-200:]

def get_web_deps():
    # DuckDuckGo search + page-to-text: used by the DuckDuckGo option and as automatic fallback
    ok, err = pip_install(["ddgs", "trafilatura"])
    return "installed" if ok else "skipped (pip failed; Parallel/Exa still work)"

def hf_token():
    try:
        from google.colab import userdata
        return (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        return ""

class Heartbeat(threading.Thread):
    """Prints a progress line every few seconds while a download/conversion runs."""
    def __init__(self, label, watch, is_dir, interval=10, total_bytes=0):
        super().__init__(daemon=True)
        self.label, self.watch, self.is_dir = label, watch, is_dir
        self.interval, self.total = interval, total_bytes
        self.t0 = time.time()
        self.stop_event = threading.Event()

    def _size(self):
        try:
            if self.is_dir:
                total = 0
                for root, _, files in os.walk(self.watch):
                    for name in files:
                        try:
                            total += os.path.getsize(os.path.join(root, name))
                        except OSError:
                            pass
                return total
            return os.path.getsize(self.watch) if os.path.exists(self.watch) else 0
        except OSError:
            return 0

    def run(self):
        while not self.stop_event.wait(self.interval):
            cur = self._size()
            el = int(time.time() - self.t0)
            msg = f"   [{self.label}] {el // 60}:{el % 60:02d} | {cur/1e9:5.1f} GB"
            if self.total:
                msg += f" / ~{self.total/1e9:.0f} GB"
            try:
                msg += f" | {shutil.disk_usage(SCRATCH).free/1e9:5.1f} GB free"
            except OSError:
                pass
            print(msg, flush=True)

    def stop(self):
        self.stop_event.set()
        self.join(timeout=5)

def run_with_progress(cmd, log, label, watch_path, is_dir, interval):
    """Run a subprocess while printing progress (file/dir size + free disk) every `interval` s."""
    with open(log, "a") as f:
        f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
        proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
    hb = Heartbeat(label, watch_path, is_dir, interval)
    hb.start()
    try:
        while proc.poll() is None:
            time.sleep(1)
    finally:
        hb.stop()
    return proc.returncode

def gpu_check():
    try:
        gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                       "--format=csv,noheader"], text=True).strip().splitlines()[0]
        name, cap, vram = [x.strip() for x in gpu.split(",")]
        print(f"✅ GPU: {name} (sm_{cap.replace('.', '')}, {vram})")
        return cap.replace(".", "")
    except Exception:
        raise RuntimeError("❌ No GPU found. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

def mount_drive():
    global USE_DRIVE_CACHE
    if USE_DRIVE_CACHE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ Google Drive mounted (build cache)")
        except Exception as e:
            USE_DRIVE_CACHE = False
            print(f"⚠️ Could not mount Google Drive ({e}); continuing without the build cache.")
    return USE_DRIVE_CACHE

def wait_jobs(jobs, t0):
    last = -1
    while not all(j.done() for j in jobs.values()):
        time.sleep(5)
        mins = int((time.time() - t0) // 60)
        if mins != last:
            last = mins
            print(f"   {mins:>2} min | " + " | ".join(f"{k}: {'done' if j.done() else '…'}" for k, j in jobs.items()))
    errors = [f"[{k}] {j.exception()}" for k, j in jobs.items()
              if j.exception() is not None and k != "web tools"]
    if errors:
        raise RuntimeError("Setup failed — just run this cell again (finished parts are reused).\n\n" + "\n\n".join(errors))


# ======================= Inkling-2-Mini specific =======================
def download_mmproj(files, token):
    """The 0.64 GB vision encoder. Never fatal: without it the model simply runs text-only."""
    if not DOWNLOAD_VISION:
        return ""
    from huggingface_hub import hf_hub_download
    name = pick_mmproj(files, MMPROJ_FILE) if files else MMPROJ_FILE
    if not name:
        print("⚠️ No vision encoder (mmproj) file found in the repo - continuing text-only.")
        return ""
    for attempt in range(3):
        try:
            return hf_hub_download(REPO_ID, name, local_dir=MODEL_DIR, token=token)
        except Exception as e:                  # noqa: BLE001
            print(f"⚠️ vision encoder download attempt {attempt + 1}/3 failed: {str(e)[:160]}")
            time.sleep(10 * (attempt + 1))
    print("⚠️ Could not download the vision encoder - continuing text-only (re-run Cell 1 to retry).")
    return ""

def download_model():
    try:
        import huggingface_hub  # noqa
    except ImportError:
        pip_install(["huggingface_hub"])
    from huggingface_hub import hf_hub_download, list_repo_files
    token = hf_token() or None
    fname, quant_used = FILENAME, QUANT
    files = []
    try:
        files = list_repo_files(REPO_ID, token=token)
        picked = pick_gguf(files, QUANT, PREFERENCE, exact_name=FILENAME)
        if not picked:
            raise RuntimeError(f"No usable GGUF quant found in {REPO_ID}. Files:\n" + "\n".join(files))
        fname, quant_used = picked
        if fname != FILENAME:
            print(f"   (no {QUANT} file in the repo — using {fname})")
    except RuntimeError:
        raise
    except Exception as e:                      # noqa: BLE001
        print(f"   (could not list repo files: {e}; trying the standard file name)")
    for old in os.listdir(MODEL_DIR):           # free disk: other quants from earlier runs
        if old.endswith(".gguf") and old != os.path.basename(fname) and not old.startswith("mmproj"):
            try:
                os.remove(os.path.join(MODEL_DIR, old))
            except OSError:
                pass
    last = None
    for attempt in range(4):
        try:
            path = hf_hub_download(REPO_ID, fname, local_dir=MODEL_DIR, token=token)
            return path, quant_used, download_mmproj(files, token)
        except Exception as e:                  # noqa: BLE001
            last = e
            print(f"⚠️ download attempt {attempt + 1}/4 failed: {str(e)[:200]}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"Download of {fname} failed after 4 tries: {last}")

def download_job():
    hb = Heartbeat("download", MODEL_DIR, True, 15, int((SIZE_GB.get(QUANT, 16.5) + 0.65) * 1e9))
    hb.start()
    try:
        return download_model()
    finally:
        hb.stop()

def check_architecture(model_path):
    """Read the GGUF header and make sure it is the merged Inkling 2 loop layout this llama.cpp tag understands."""
    try:
        meta, tensors, _ = read_gguf_info(model_path)
    except Exception as e:                      # noqa: BLE001
        return f"(could not read the GGUF header: {str(e)[:80]})"
    arch = meta.get("general.architecture", "?")
    msg = f"{arch}, {meta.get(f'{arch}.block_count', '?')} layers, {len(tensors)} tensors"
    if not str(arch).startswith("inkling2"):
        msg += "  ⚠️ unexpected architecture - the server may refuse to load this file"
    return msg

def main():
    global ARCH, USE_CACHE, CACHED_BIN, CACHED_QUANT
    mount_drive()
    USE_CACHE = USE_DRIVE_CACHE
    ARCH = gpu_check()
    free_gb = shutil.disk_usage(SCRATCH).free / 1e9
    if free_gb < 32:
        print(f"⚠️ Only {free_gb:.0f} GB of disk free; the build + model need ~25 GB.")
    if USE_CACHE:
        os.makedirs(CACHE_DIR, exist_ok=True)
    CACHED_BIN   = f"{CACHE_DIR}/llama-server-{LLAMA_TAG}-sm{ARCH}"
    CACHED_QUANT = f"{CACHE_DIR}/llama-quantize-{LLAMA_TAG}-sm{ARCH}"

    t0 = time.time()
    print(f"⏳ Building llama.cpp {LLAMA_TAG} and downloading {REPO_ID} ({QUANT}) at the same time...")
    with ThreadPoolExecutor(max_workers=4) as ex:
        jobs = {"build": ex.submit(build_job), "download": ex.submit(download_job),
                "tunnel tool": ex.submit(get_cloudflared), "web tools": ex.submit(get_web_deps)}
        wait_jobs(jobs, t0)
    build_status = jobs["build"].result()
    model_path, quant_used, mmproj_path = jobs["download"].result()
    cf_status = jobs["tunnel tool"].result()
    web_status = jobs["web tools"].result()
    try:
        ui_status = prepare_webui()
    except Exception as e:                      # noqa: BLE001
        ui_status = f"will be fetched by Cell 2 ({str(e)[:80]})"

    cfg = {"model": model_path, "quant": quant_used, "bin": BIN_PATH, "cloudflared": CF_PATH,
           "tag": LLAMA_TAG, "ui_tag": LLAMA_TAG, "webui_src": WEBUI_SRC, "mmproj": mmproj_path}
    json.dump(cfg, open(CONFIG, "w"))
    print(f"\n✅ Server:      {build_status} (llama.cpp {LLAMA_TAG})")
    print(f"✅ Model:       {os.path.basename(model_path)} ({os.path.getsize(model_path)/1e9:.1f} GB, {quant_used})")
    print(f"✅ Architecture: {check_architecture(model_path)}")
    print("✅ Vision:      " + (f"{os.path.basename(mmproj_path)} ({os.path.getsize(mmproj_path)/1e9:.2f} GB)" if mmproj_path else "not downloaded (text-only)"))
    print(f"✅ Tunnel tool: {cf_status}")
    print(f"✅ Web tools:   {web_status}")
    print(f"✅ Chat UI:     {ui_status}")
    print(f"⏱  {int(time.time() - t0)//60} min {int(time.time() - t0)%60} s.  👉 Now run Cell 2.")

if __name__ == "__main__":
    main()


In [ ]:
#@title 2️⃣ Launch — choose context, reasoning and web search, then run (keep this cell running!) { display-mode: "form" }
VISION = True #@param {type:"boolean"}
CONTEXT_WINDOW = "4096 · small and fast (default)" #@param ["4096 · small and fast (default)", "8192", "16384", "32768", "65536", "131072", "262144", "Max · the biggest window that fits this GPU (up to 1M)"]
KV_CACHE = "auto · best precision that fits the chosen context (default)" #@param ["auto · best precision that fits the chosen context (default)", "f16 · full precision", "q8_0 · 8-bit: less memory", "q4_0 · 4-bit: the least memory"]
REASONING = "Max · highest effort (0.99), unlimited thinking (default)" #@param ["Max · highest effort (0.99), unlimited thinking (default)", "High · effort 0.9", "Medium · effort 0.7", "Low · effort 0.2", "Minimal · effort 0.1", "None · effort 0 (the model still thinks briefly)"]
WEB_SEARCH = "Parallel · free, no key needed (default)" #@param ["Parallel · free, no key needed (default)", "Exa · free tier is rate-limited, best with a key", "DuckDuckGo · free, no key needed", "Off · no web tools"]
AUTO_FALLBACK = True #@param {type:"boolean"}
PARALLEL_API_KEY = "VtYAbqEv6_sOvxUtcGeHI28jHmIpGuxM_talrLr6" #@param {type:"string"}
EXA_API_KEY = "29334ebe-08ca-4c98-8b57-af84986d1464" #@param {type:"string"}
PUBLIC_LINK = "Cloudflare · no account, new link each run (default)" #@param ["Cloudflare · no account, new link each run (default)", "Off · private Colab link only"]
SHOW_INLINE = True #@param {type:"boolean"}
# CONTEXT_WINDOW: 4,096 tokens by default (the model itself supports 1M). Max reasoning writes a lot of thinking, so for anything
#   non-trivial pick 16384 or more; "Max" = the biggest window that physically fits this GPU. The chat memory (KV cache) is small,
#   but llama.cpp also needs working buffers that grow with the window, so big windows automatically use a smaller prompt batch
#   (slower prompt reading). On a free T4 the practical range is roughly 32K-128K depending on the quant.
# KV_CACHE: the chat memory (KV cache). 8-bit is almost lossless; 4-bit doubles the context but is a bit less precise.
#   If this llama.cpp build cannot quantize the chat memory for this model, 16-bit is used automatically.
# REASONING: Inkling reasons natively. The effort is a number the chat template puts in the prompt ("Thinking effort level: 0.99"):
#   Max = the template's "max" (0.99) with no token limit; the lower levels think less. "None" still thinks briefly (the model card says so).
# AUTO_FALLBACK: if the chosen search service is down or rate-limited, the other free ones are tried automatically.
# API keys are hardcoded above (source of truth). Colab secrets named PARALLEL_API_KEY / EXA_API_KEY are only
#   used if the field above is emptied. A Parallel key raises limits and adds a deep "research" tool.
# PUBLIC_LINK: anyone with the link can use your session. Cloudflare needs no account; the address changes each run.
# SHOW_INLINE: also shows the chat right here in the notebook (private, always works).

# ---------------- advanced (rarely needed) ----------------
TEMP, TOP_P, TOP_K, MIN_P = 1.0, 1.0, 0, 0.0     # the vendor's sampling: temperature 1.0, no top-k / min-p (top_k 0 = off)
REPEAT_PENALTY   = 1.0                           # the model card uses no repetition penalty
PRESENCE_PENALTY = 0.0                           # if reasoning ever loops, try 0.5
MAX_TOKENS       = 32768                         # longest single reply the chat UI asks for. The model card warns about endless
                                                 # reasoning loops, so this is the safety valve; -1 = unlimited
THINKING_BUDGET  = -1                            # > 0 adds a hard cap on thinking tokens (best effort, e.g. 8192); -1 = none
IMAGE_MAX_TOKENS = 0                             # 0 = let the vision encoder decide; e.g. 1024 caps the size of an image
UBATCH           = 0                             # 0 = automatic: 512 for small windows, smaller for big ones (their working buffers grow with the window)
HOST_EXPERT_LAYERS = -1                          # -1 = automatic; 0 = never use system RAM; N = put the experts of N outer layers in RAM
CTX_CHECKPOINTS  = 4                             # prompt checkpoints kept in RAM (llama.cpp's default 32 would eat Colab's RAM)
CACHE_RAM_MIB    = 768                           # RAM for remembering other chats' prompts (llama.cpp's default is 8192)
HTTP_THREADS     = 32                            # web-server threads. llama.cpp's default is only 3 on a 2-core Colab: a streaming reply,
                                                 # a tool call and a stale connection could block 'new chat' (shows up as 502 / Error 500)
SHOW_THOUGHT_LIVE = False                        # False = the reasoning panel stays collapsed while it streams (much smoother in long
                                                 # Max-reasoning chats; click it to read). True = watch the thoughts stream live
MIN_FREE_MIB     = 350                           # GPU memory that must stay free after loading, so long chats never crash later
MAX_SERVER_RESTARTS = 5                          # restarts allowed within 30 minutes before giving up (a crash an hour does not count)
# -----------------------------------------------------------
import os, re, sys, json, time, datetime, subprocess, shutil, socket, threading
import requests
from IPython.display import display, HTML

PROFILE = dict(
    NAME="Inkling-2-Mini-Uncensored",
    N_LAYER=24, LOOP_START=6, LOOP_END=18, LOOP_COUNT=3,   # 24 stored layers; layers 6..17 run three times (48 layer visits per token)
    N_KV_HEAD=8, HEAD_DIM=96,                        # 24 query heads / 8 KV heads, hidden size 2304 (re-read from the GGUF when it has them)
    CTX_LADDER=[1048576, 524288, 262144, 196608, 131072, 98304, 65536, 49152, 32768, 16384, 8192, 4096],
    KV_AUTO_ORDER=["f16", "q8_0", "q4_0"],           # best precision first (after the fewest layers in RAM); Max still means the biggest window
    OVERHEAD_MIB=700,                                # CUDA context + the 40 sliding-window caches (1,024 tokens each); the working buffers are added per window size
    VISION_MIB=1100,                                 # vision encoder weights (0.64 GB) + its compute buffer
    CONTEXT_SHIFT=False,                             # sliding-window + looped model: no context shifting
    CONFIG="/content/inkling2_mini_config.json",
    IDENTITY="You are Inkling-2-Mini (uncensored, AtomicChat build), a capable multimodal assistant that can read images, use web tools and think carefully before answering. Today's date is {today}.",
)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, json, struct

# ---------------------------------------------------------------- minimal GGUF header reader
_SCALARS = {0: "<B", 1: "<b", 2: "<H", 3: "<h", 4: "<I", 5: "<i", 6: "<f", 7: "<?", 10: "<Q", 11: "<q", 12: "<d"}
_KEEP_SUFFIX = (".block_count", ".context_length", ".head_count_kv", ".head_count", ".embedding_length", ".key_length",
                ".loop.start", ".loop.end", ".loop.count", ".loop_start", ".loop_end", ".loop_count",
                ".core_start", ".core_end", ".num_passes", ".sliding_window", ".expert_count", ".expert_used_count")

def read_gguf_info(path, want_prefixes=("general.", "tokenizer.chat_template"), buf_mb=192):
    """-> (meta, tensors, data_start).
    meta    : {key: value} for the small keys (scalars + strings); big arrays are skipped.
    tensors : [(name, offset_in_data_section)] for every tensor.
    Pure python, no dependency. Raises ValueError if the file is not a GGUF."""
    with open(path, "rb") as f:
        buf = f.read(buf_mb << 20)
    if buf[:4] != b"GGUF":
        raise ValueError("not a GGUF file")
    ver = struct.unpack_from("<I", buf, 4)[0]
    if ver < 2:
        raise ValueError(f"unsupported GGUF version {ver}")
    n_tensors, n_kv = struct.unpack_from("<QQ", buf, 8)
    pos, out = 24, {}

    def rstr(p):
        n = struct.unpack_from("<Q", buf, p)[0]
        return buf[p + 8:p + 8 + n].decode("utf-8", "replace"), p + 8 + n

    def skip(t, p):
        if t in _SCALARS:
            return p + struct.calcsize(_SCALARS[t])
        if t == 8:
            n = struct.unpack_from("<Q", buf, p)[0]
            return p + 8 + n
        if t == 9:
            et, cnt = struct.unpack_from("<IQ", buf, p)
            p += 12
            if et in _SCALARS:
                return p + cnt * struct.calcsize(_SCALARS[et])
            for _ in range(cnt):
                p = skip(et, p)
            return p
        raise ValueError(f"bad GGUF value type {t}")

    for _ in range(n_kv):
        key, pos = rstr(pos)
        t = struct.unpack_from("<I", buf, pos)[0]
        pos += 4
        keep = key.startswith(want_prefixes) or key.endswith(_KEEP_SUFFIX)
        if keep and t in _SCALARS:
            out[key] = struct.unpack_from(_SCALARS[t], buf, pos)[0]
            pos += struct.calcsize(_SCALARS[t])
        elif keep and t == 8:
            out[key], pos = rstr(pos)
        else:
            pos = skip(t, pos)
    tensors = []
    for _ in range(n_tensors):
        name, pos = rstr(pos)
        n_dims = struct.unpack_from("<I", buf, pos)[0]
        pos += 4 + 8 * n_dims + 4                       # n_dims, the dims, the ggml type
        off = struct.unpack_from("<Q", buf, pos)[0]
        pos += 8
        tensors.append((name, off))
    align = out.get("general.alignment") or 32
    align = int(align) if isinstance(align, (int, float)) and align > 0 else 32
    return out, tensors, (pos + align - 1) // align * align

def read_gguf_meta(path, **kw):
    return read_gguf_info(path, **kw)[0]

def expert_bytes_by_layer(tensors, data_start, file_size):
    """{layer: bytes of that layer's routed-expert tensors} (blk.N.ffn_*_exps*), sizes taken from the offset gaps."""
    order = sorted(tensors, key=lambda t: t[1])
    out = {}
    for i, (name, off) in enumerate(order):
        end = order[i + 1][1] if i + 1 < len(order) else file_size - data_start
        m = re.match(r"blk\.(\d+)\.ffn_.*_exps", name)
        if m:
            out[int(m.group(1))] = out.get(int(m.group(1)), 0) + max(0, end - off)
    return out


# ---------------------------------------------------------------- the looped architecture
def arch_facts(meta, profile):
    """Layer layout from the GGUF (falls back to the profile): n_layer, loop_start, loop_end, loop_count,
    kv_heads, head_dim, ctx_train."""
    arch = meta.get("general.architecture", "") or ""

    def pos_num(*keys):
        for k in keys:
            v = meta.get(f"{arch}.{k}")
            if isinstance(v, (int, float)) and not isinstance(v, bool) and v > 0:
                return int(v)
        return 0

    def any_num(suffixes):
        for k, v in meta.items():
            if k.endswith(suffixes) and isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0:
                return int(v)
        return None

    n_layer = pos_num("block_count") or profile["N_LAYER"]
    ls = any_num((".loop.start", ".loop_start", ".core_start"))
    le = any_num((".loop.end", ".loop_end", ".core_end"))
    lc = any_num((".loop.count", ".loop_count", ".num_passes"))
    ls = profile["LOOP_START"] if ls is None else ls
    le = profile["LOOP_END"] if le is None else le
    lc = profile["LOOP_COUNT"] if not lc else lc
    if not (0 <= ls < le <= n_layer):
        ls, le = profile["LOOP_START"], profile["LOOP_END"]
    kvh = pos_num("attention.head_count_kv") or profile["N_KV_HEAD"]
    hd = pos_num("attention.key_length")
    hd = hd if 32 <= hd <= 512 else profile["HEAD_DIM"]
    return n_layer, ls, le, lc, kvh, hd, pos_num("context_length")

def full_attn_visits(n_layer, loop_start, loop_end, loop_count, period=6):
    """Layer visits per token that keep a full-length KV cache: every 6th layer is global attention, and the layers
    of the loop are visited `loop_count` times, each visit with its own cache (the other visits use a 1,024-token window)."""
    n = 0
    for i in range(n_layer):
        if (i + 1) % period == 0:
            n += loop_count if loop_start <= i < loop_end else 1
    return n

def host_eligible_layers(n_layer, loop_start, loop_end):
    """Layers whose experts may live in system RAM, in the order they are moved there. The looped layers
    (loop_start..loop_end-1) are NEVER eligible: with llama.cpp b126xx their host-resident experts give garbage output."""
    return list(range(n_layer - 1, loop_end - 1, -1)) + list(range(loop_start - 1, -1, -1))

def host_expert_regex(layers):
    """llama-server --override-tensor argument that puts the routed experts of `layers` in system RAM."""
    return r"blk\.(" + "|".join(str(i) for i in sorted(layers)) + r")\.ffn_.*_exps=CPU"


# ---------------------------------------------------------------- GPU memory planner
KV_BYTES_PER_ELEM = {"f16": 2.0, "q8_0": 34 / 32, "q4_0": 18 / 32}   # ggml block sizes

def kv_mib(ctx, kv, n_layer, n_kv_head, head_dim):
    return 2 * n_layer * n_kv_head * head_dim * KV_BYTES_PER_ELEM[kv] * ctx / 1048576

# The attention of this model is not run through flash-attention in llama.cpp, so the working buffers hold a score matrix
# of (context x prompt batch x 24 heads) floats: measured 25,535 MiB at 262,144 tokens with the default batch of 512.
# So the prompt batch (-ub) has to shrink as the window grows (slower prompt reading, but the window fits).
UB_LADDER = (512, 256, 128, 64, 32, 16)

def compute_mib(ctx, ub):
    """GPU working buffers: ~1.5 MiB per batch token + ~200 bytes per (context token x batch token)."""
    return 1.5 * ub + ctx * ub * 200 / 1048576

def pick_ubatch(ctx, cap_mib=1200):
    """Largest prompt batch whose working buffers stay under cap_mib at this context size."""
    for ub in UB_LADDER:
        if compute_mib(ctx, ub) <= cap_mib:
            return ub
    return UB_LADDER[-1]

def plan_candidates(model_mib, total_mib, ladder, target, kv_pref, dims, overhead_mib, min_free_mib,
                    auto_order=None, host_mib=(), force_k=-1, ub_user=0, max_extra_k=None):
    """Ordered (ctx, kv, k, ub) launch attempts; k = number of layers whose experts go to system RAM, ub = prompt batch.
    Largest context first; at equal context the fewest host layers first, then the best KV precision.
    Entries predicted to fit come first; the rest (a safety net if the estimate is too pessimistic) follow.
    host_mib[i] = GPU MiB freed by moving the i-th eligible layer's experts to RAM. force_k >= 0 pins k, ub_user > 0 pins ub.
    max_extra_k (used for "Max"): never use more than that many host layers beyond what the weights alone need."""
    kv_order = {"auto": auto_order or ["q8_0", "q4_0"], "f16": ["f16"], "q8_0": ["q8_0"], "q4_0": ["q4_0"]}[kv_pref]
    sizes = [c for c in ladder if c <= target]
    kmax = len(host_mib)
    fit, nofit = [], []
    k_base = 0
    if max_extra_k is not None and sizes:                  # host layers needed for the weights at the smallest window
        c0 = sizes[-1]
        need0 = (model_mib + kv_mib(c0, kv_order[-1], *dims) + compute_mib(c0, ub_user if ub_user > 0 else pick_ubatch(c0))
                 + overhead_mib + min_free_mib)
        freed0 = 0.0
        while need0 - freed0 > total_mib and k_base < kmax:
            freed0 += host_mib[k_base]
            k_base += 1
    for c in sizes:
        ub = ub_user if ub_user > 0 else pick_ubatch(c)
        for kv in kv_order:
            need = model_mib + kv_mib(c, kv, *dims) + compute_mib(c, ub) + overhead_mib + min_free_mib
            k, freed = 0, 0.0
            if force_k >= 0:
                k = min(force_k, kmax)
                freed = sum(host_mib[:k])
            else:
                while need - freed > total_mib and k < kmax:
                    freed += host_mib[k]
                    k += 1
            ok = need - freed <= total_mib and (max_extra_k is None or k <= k_base + max_extra_k)
            ub_c = ub
            if ok and ub_user <= 0:                     # spare GPU memory -> a bigger prompt batch (faster reading of long chats)
                slack = total_mib - (need - freed)
                for cand in UB_LADDER:
                    if cand <= ub:
                        break
                    if compute_mib(c, cand) - compute_mib(c, ub) <= 0.8 * slack:
                        ub_c = cand
                        break
            (fit if ok else nofit).append((c, kv, k, ub_c))
    fit.sort(key=lambda t: (-t[0], t[2], kv_order.index(t[1])))
    kk = min(force_k, kmax) if force_k >= 0 else kmax
    return fit + [(c, kv, kk, ub) for (c, kv, _k, ub) in nofit if kv == kv_order[-1]][:3]

def tool_cap_chars(ctx):
    """Max characters one web-tool result may add to the conversation (~1/8 of the window, 3 chars/token)."""
    return int(min(30000, max(2500, ctx * 0.375)))


# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- choose a GGUF file in a repo listing
def pick_gguf(files, quant, preference, exact_name=None):
    """-> (filename, quant_used) or None. Exact name first, then the requested quant, then the next best.
    Vision encoders (mmproj) and LoRA adapters are never picked as the model."""
    ggufs = sorted(f for f in files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                   and "lora" not in f.lower() and "/" not in f and not re.search(r"-\d{5}-of-\d{5}", f))
    if exact_name and exact_name in ggufs:
        return exact_name, quant
    order = preference[preference.index(quant):] if quant in preference else list(preference)
    for q in order:
        pat = re.compile(r"(?<![A-Za-z0-9])" + re.escape(q) + r"(?![A-Za-z0-9])", re.I)
        hit = [f for f in ggufs if pat.search(f)]
        if hit:
            return hit[0], q
    return None


# ---------------------------------------------------------------- tiny PNG for the vision self-test
def make_png_solid(w, h, rgb):
    """A valid w x h solid-colour PNG (no imaging library needed)."""
    import zlib
    def chunk(tag, data):
        c = struct.pack(">I", len(data)) + tag + data
        return c + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
    row = b"\x00" + bytes(rgb) * w
    raw = row * h
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def pick_mmproj(files, preferred=""):
    """Vision-encoder GGUF in a repo listing: the preferred name, else F16 > BF16 > F32 > anything called mmproj*
    (F16 first: a T4 has no native BF16)."""
    cands = sorted(f for f in files if f.lower().startswith("mmproj") and f.lower().endswith(".gguf") and "/" not in f)
    if preferred and preferred in cands:
        return preferred
    for pat in (r"(?<![a-z])f16", r"bf16", r"f32"):
        for f in cands:
            if re.search(pat, f.lower()):
                return f
    return cands[0] if cands else ""
# ==== PURE-END ====


LOG, TLOG = "/content/llama-server.log", "/content/tunnel.log"
CONFIG = PROFILE["CONFIG"]
WEB_SCRIPT, MCP_CONFIG = "/content/colab_web_mcp.py", "/content/colab_mcp.json"
WEBUI_DIR = "/content/webui"
MiB = 1048576
LOCAL = requests.Session()
LOCAL.trust_env = False       # talk to the local server/tunnel directly, never through a proxy

def free_port(start):
    """First port from `start` that nothing else is using."""
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sk:
            try:
                sk.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"No free port found near {start}")

def secret(name, typed):
    if typed.strip():
        return typed.strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

def gpu_mem():
    """(used MiB, total MiB) summed over GPUs, or (None, None)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
        used = total = 0
        for line in out.strip().splitlines():
            u, t = line.split(",")
            used += int(u); total += int(t)
        return (used, total) if total else (None, None)
    except Exception:
        return None, None

TOGGLE_JS = r'''/* Injected by the notebook (tpuchat-v2): "Allow tool calls" switch inside the chat box,
   right after the + button. Applies instantly - never reloads the page, so a running
   answer (text or reasoning) is never interrupted. The choice is saved per browser. */
(function () {
  "use strict";
  if (window.__tpuchatAllowTools) return;
  window.__tpuchatAllowTools = true;

  var STATE_KEY = "tpuchat.allowTools";
  var TOOL_NAMES = ["web_search", "web_fetch", "web_research"];
  /* localStorage lists the official UI reads exactly once, at startup */
  var STORE_KEYS = ["LlamaUi.alwaysAllowedTools", "LlamaCppWebui.alwaysAllowedTools"];
  /* permission keys the UI derives for these tools (toolKey(): server:/mcp-<id>:/custom:) */
  var PERM_KEYS = [];
  var JUNK_KEYS = [];
  TOOL_NAMES.forEach(function (n) {
    PERM_KEYS.push("server:" + n, "mcp-web:" + n, "mcp-web:" + n.replace(/^web_/, ""), "custom:" + n);
  });
  ["web_web_search", "web_web_fetch", "web_web_research"].forEach(function (n) {
    JUNK_KEYS.push("server:" + n, "mcp-web:" + n, "custom:" + n);
  });
  TOOL_NAMES.forEach(function (n) {
    JUNK_KEYS.push("browser:" + n, "mcp:" + n);
  });

  function isOn() {
    try { var v = localStorage.getItem(STATE_KEY); return v === null ? true : v === "1"; }
    catch (e) { return true; }
  }
  function readList(k) {
    try {
      var a = JSON.parse(localStorage.getItem(k) || "[]");
      return Array.isArray(a) ? a.filter(function (x) { return typeof x === "string"; }) : [];
    } catch (e) { return []; }
  }
  function writeList(k, list) {
    try { localStorage.setItem(k, JSON.stringify(list)); } catch (e) {}
  }
  function cleanAndWrite(k, set) {
    JUNK_KEYS.forEach(function (p) { delete set[p]; });
    writeList(k, Object.keys(set));
  }
  function syncPerms(on) {
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      PERM_KEYS.forEach(function (p) { if (on) set[p] = 1; else delete set[p]; });
      cleanAndWrite(k, set);
    });
  }
  function allowSaved(tool) {
    var keys = ["server:" + tool, "mcp-web:" + tool, "mcp-web:" + tool.replace(/^web_/, ""), "custom:" + tool];
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      keys.forEach(function (p) { set[p] = 1; });
      cleanAndWrite(k, set);
    });
  }
  /* runs before the UI boots: the app picks this list up when it initializes */
  syncPerms(isOn());

  /* -------------- live plumbing (no reload: streaming is never interrupted) -------------- */

  var sessionAll = false;            /* "allow this session" covers all web tools until OFF again */
  var seenCard = { tool: "", t: 0 }; /* built-in permission card the user may act on */

  function isWebTool(t) { return typeof t === "string" && /^web_/.test(t); }

  /* Fetch gate - active only while the switch is OFF. Covers the one case localStorage
     cannot: web tools the UI still holds in its in-memory allowed set from an earlier
     boot, which would otherwise run silently instead of asking. */
  var origFetch = window.fetch.bind(window);
  window.fetch = function () {
    try {
      var input = arguments[0], init = arguments[1];
      var url = typeof input === "string" ? input : (input && input.url) || "";
      var method = ((init && init.method) || (input && input.method) || "GET").toUpperCase();
      if (method === "POST" && /\/tools\/?$/.test(String(url).split("?")[0]) &&
          init && typeof init.body === "string") {
        var tool = null;
        try { tool = (JSON.parse(init.body) || {}).tool; } catch (e) {}
        if (isWebTool(tool) && !isOn() && !sessionAll &&
            !(seenCard.tool === tool && Date.now() - seenCard.t < 8000)) {
          return gate(tool, arguments);
        }
      }
    } catch (e) { /* never break the app */ }
    return origFetch.apply(null, arguments);
  };

  var activeFinish = null;   /* decision callback of the currently open prompt */
  var gateChain = Promise.resolve();
  function gate(tool, args) {
    var run = function () {
      return new Promise(function (resolve) {
        var settled = false;
        var signal = args[1] && args[1].signal;
        function finish(allow, remember) {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          if (remember) sessionAll = true;
          if (remember === "saved") allowSaved(tool);
          if (allow) resolve(origFetch.apply(null, args));
          else resolve(new Response(JSON.stringify({
                   error: "Blocked by the user: the Allow tool calls switch is OFF in the chat box. Turn it ON (green) to run web tools without asking."
                 }), { status: 403, statusText: "Blocked",
                      headers: { "Content-Type": "application/json" } }));
        }
        function abortNow() {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          resolve(Promise.reject(new DOMException("The tool call was aborted.", "AbortError")));
        }
        if (signal) {
          if (signal.aborted) { abortNow(); return; }
          signal.addEventListener("abort", abortNow, { once: true });
        }
        showGateCard(tool, finish);
      });
    };
    gateChain = gateChain.then(run, run);
    return gateChain;
  }

  function esc(s) {
    return String(s).replace(/[&<>"']/g, function (c) {
      return { "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c];
    });
  }

  var gateEl = null;
  function showGateCard(tool, finish) {
    hideGateCard();
    var d = document.createElement("div");
    d.id = "tpu-gate-card";
    d.setAttribute("role", "dialog");
    d.setAttribute("aria-label", "Tool call permission");
    d.innerHTML = '<div class="t">The model wants to run <b>' + esc(tool) + "</b></div>" +
      '<div class="s">The "Allow tool calls" switch is OFF, so this call needs your OK.</div>' +
      '<div class="row">' +
      '<button type="button" data-a="once">Allow once</button>' +
      '<button type="button" data-a="session">Allow this session</button>' +
      '<button type="button" data-a="saved">Always allow</button>' +
      '<button type="button" data-a="deny" class="deny">Deny</button>' +
      "</div>";
    d.addEventListener("click", function (e) {
      var b = e.target && e.target.closest ? e.target.closest("button[data-a]") : null;
      if (!b) return;
      var a = b.getAttribute("data-a");
      finish(a !== "deny", a === "session" ? "session" : (a === "saved" ? "saved" : null));
    });
    (document.body || document.documentElement).appendChild(d);
    gateEl = d;
    activeFinish = finish;
  }
  function hideGateCard() {
    if (gateEl) { try { gateEl.remove(); } catch (e) {} gateEl = null; }
  }
  document.addEventListener("keydown", function (e) {
    if (e.key === "Escape" && activeFinish) activeFinish(false, null);
  }, true);

  /* Watch the built-in permission card ("Allow use of <tool> ..."). While the switch is
     ON, approve it automatically (only needed when the in-memory set is missing keys,
     e.g. the switch was turned ON mid-session). While OFF, just remember it so a call
     the user just approved by hand is never asked twice. */
  function scanForCard() {
    var btns = document.getElementsByTagName("button");
    for (var i = 0; i < btns.length; i++) {
      var b = btns[i];
      if ((b.textContent || "").trim() !== "Allow once") continue;
      var root = b.parentElement, cardText = "";
      for (var k = 0; k < 7 && root; k++, root = root.parentElement) {
        cardText = root.textContent || "";
        if (cardText.indexOf("Allow use of ") >= 0) break;
      }
      if (cardText.indexOf("Allow use of ") < 0) continue;
      var m = cardText.match(/Allow use of\s+([A-Za-z0-9_.-]+)/);
      if (m) { seenCard.tool = m[1]; seenCard.t = Date.now(); }
      if (isOn() && !b.dataset.tpuDone) {
        b.dataset.tpuDone = "1";
        try { b.click(); } catch (e) {}
      }
    }
  }

  /* -------------- the switch, rendered inside the composer after the + button -------------- */

  var CSS_TEXT =
    "#tpu-allow-tools{display:inline-flex;align-items:center;gap:7px;height:32px;padding:0 12px;" +
    "border-radius:999px;border:1px solid rgba(148,163,184,.35);background:rgba(100,116,139,.14);" +
    "color:inherit;opacity:.85;font:600 12px/1 ui-sans-serif,system-ui,-apple-system,sans-serif;" +
    "cursor:pointer;user-select:none;white-space:nowrap;flex:0 0 auto;" +
    "transition:border-color .15s,background .15s,opacity .15s}" +
    "#tpu-allow-tools:hover{opacity:1;border-color:rgba(148,163,184,.6)}" +
    "#tpu-allow-tools .dot{width:8px;height:8px;border-radius:50%;background:#64748b;flex:0 0 auto;" +
    "transition:background .15s,box-shadow .15s}" +
    "#tpu-allow-tools.on{border-color:rgba(34,197,94,.55);background:rgba(34,197,94,.13);opacity:1}" +
    "#tpu-allow-tools.on .dot{background:#22c55e;box-shadow:0 0 7px rgba(34,197,94,.8)}" +
    "#tpu-allow-tools.on .lbl{color:#16a34a}" +
    "#tpu-allow-tools .lbl-short{display:none}" +
    "#tpu-gate-card{position:fixed;left:50%;bottom:96px;transform:translateX(-50%);z-index:2147483000;" +
    "width:min(440px,calc(100vw - 24px));padding:14px 16px;border-radius:14px;color:#e2e8f0;" +
    "background:#0f172a;border:1px solid rgba(148,163,184,.3);box-shadow:0 12px 40px rgba(0,0,0,.45);" +
    "font:400 13px/1.5 ui-sans-serif,system-ui,-apple-system,sans-serif;animation:tpu-pop .16s ease-out}" +
    "#tpu-gate-card .t{font-weight:600;margin-bottom:2px}" +
    "#tpu-gate-card .s{opacity:.75;margin-bottom:10px}" +
    "#tpu-gate-card .row{display:flex;flex-wrap:wrap;gap:8px}" +
    "#tpu-gate-card button{all:unset;cursor:pointer;padding:6px 12px;border-radius:8px;" +
    "font:600 12px/1 ui-sans-serif,system-ui,sans-serif;color:inherit;" +
    "border:1px solid rgba(148,163,184,.4);background:rgba(148,163,184,.12)}" +
    "#tpu-gate-card button:hover{background:rgba(148,163,184,.24)}" +
    "#tpu-gate-card button.deny{border-color:rgba(239,68,68,.5);color:#f87171}" +
    "@keyframes tpu-pop{from{opacity:0;transform:translateX(-50%) translateY(6px)}" +
    "to{opacity:1;transform:translateX(-50%) translateY(0)}}" +
    "@media (prefers-color-scheme: dark){#tpu-allow-tools.on .lbl{color:#4ade80}}" +
    "@media (prefers-color-scheme: light){#tpu-gate-card{background:#fff;color:#0f172a;" +
    "border-color:#cbd5e1;box-shadow:0 12px 40px rgba(15,23,42,.18)}}" +
    "@media (max-width:640px){#tpu-allow-tools{height:28px;padding:0 9px;gap:5px;font-size:11px}" +
    "#tpu-allow-tools .lbl-long{display:none}#tpu-allow-tools .lbl-short{display:inline}" +
    "#tpu-gate-card{bottom:84px}}";

  var chip = null;
  function paint() {
    if (!chip) return;
    var on = isOn();
    chip.classList.toggle("on", on);
    chip.setAttribute("aria-checked", on ? "true" : "false");
    var long = chip.querySelector(".lbl-long"), short = chip.querySelector(".lbl-short");
    if (long) long.textContent = "Allow tool calls: " + (on ? "ON" : "OFF");
    if (short) short.textContent = "Tools: " + (on ? "ON" : "OFF");
  }
  function buildChip() {
    try {
      var css = document.createElement("style");
      css.id = "tpu-allow-tools-css";
      css.textContent = CSS_TEXT;
      document.head.appendChild(css);
    } catch (e) {}
    var b = document.createElement("button");
    b.id = "tpu-allow-tools";
    b.type = "button";
    b.setAttribute("role", "switch");
    b.setAttribute("aria-label", "Allow tool calls");
    b.title = "ON: web tools (search / fetch) run without asking. OFF: each web tool call asks first. " +
      "Applies instantly - no reload, typing and generating continue undisturbed.";
    b.innerHTML = '<span class="dot"></span><span class="lbl lbl-long"></span><span class="lbl lbl-short"></span>';
    /* keep the composer textarea focused and never trigger the + menu or a form submit */
    ["mousedown", "pointerdown", "touchstart", "touchend"].forEach(function (ev) {
      b.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); });
    });
    b.addEventListener("click", function (e) {
      e.preventDefault();
      e.stopPropagation();
      var next = !isOn();
      try { localStorage.setItem(STATE_KEY, next ? "1" : "0"); } catch (err) {}
      syncPerms(next);
      if (!next) sessionAll = false;   /* a fresh OFF period asks again */
      paint();
      /* turning ON while a call is waiting for permission: let it through */
      if (next && activeFinish) activeFinish(true, "session");
      try {
        window.dispatchEvent(new StorageEvent("storage", { key: STATE_KEY, newValue: next ? "1" : "0" }));
      } catch (err) {}
    });
    paint();
    return b;
  }
  function ensureChip() {
    var host = document.querySelector("button.file-upload-button");
    if (!host || !host.isConnected) return false;
    if (!chip) chip = buildChip();
    /* preferred spot: the left group that holds the + button, next to it */
    var group = host.closest(".mr-auto") || host.parentElement;
    if (!group) return false;
    if (chip.isConnected && chip.parentElement === group) return true;
    try { group.appendChild(chip); paint(); } catch (e) { return false; }
    return true;
  }

  window.addEventListener("storage", function (e) {
    if (e && e.key === STATE_KEY) { try { paint(); } catch (err) {} }
  });

  /* -------------- keep-alive: survive Svelte re-renders and route changes -------------- */

  var scanTimer = 0, chipTimer = 0;
  function scheduleScan() {
    if (scanTimer) return;
    scanTimer = setTimeout(function () { scanTimer = 0; try { scanForCard(); } catch (e) {} }, 120);
  }
  function scheduleChip() {
    if (chipTimer) return;
    chipTimer = setTimeout(function () { chipTimer = 0; try { ensureChip(); } catch (e) {} }, 150);
  }
  function start() {
    try {
      var mo = new MutationObserver(function (muts) {
        scheduleChip();
        for (var i = 0; i < muts.length; i++) {
          var nodes = muts[i].addedNodes;
          for (var j = 0; j < nodes.length; j++) {
            var n = nodes[j];
            if (n.nodeType === 1 && (n.textContent || "").indexOf("Allow use of ") >= 0) {
              scheduleScan();
              return;
            }
          }
        }
      });
      mo.observe(document.documentElement, { childList: true, subtree: true });
    } catch (e) {}
    setInterval(function () {
      try { ensureChip(); } catch (e) {}
      try { scanForCard(); } catch (e) {}
    }, 1500);
    try { ensureChip(); } catch (e) {}
    console.info("[tpuchat] Allow tool calls switch installed (in the chat box, next to +)");
  }
  if (document.readyState === "loading") document.addEventListener("DOMContentLoaded", start);
  else start();
})();'''


WEB_MCP_SOURCE = r'''#!/usr/bin/env python3
"""Colab web tools: a tiny stdio MCP server for llama-server (--mcp-servers-config).

Gives the model web_search / web_fetch (and web_research when a Parallel key is set), backed by:
  parallel   - Parallel MCP (https://mcp.parallel.ai/mcp), free without a key
  exa        - Exa MCP (https://mcp.exa.ai/mcp), free tier is rate limited, key optional
  duckduckgo - DuckDuckGo search (ddgs) + direct page download, no key
If the chosen backend fails (rate limit, network, timeout) the next one is tried, and EVERY call is
bounded by a hard deadline (WEB_DEADLINE, default 80 s): a slow backend can never leave the chat
hanging, and the answer always comes back before a proxy (Cloudflare: 100 s) would cut the request.

Settings come from environment variables set by the notebook:
  WEB_BACKEND        parallel | exa | duckduckgo        (default parallel)
  PARALLEL_API_KEY   optional, higher limits + adds web_research
  EXA_API_KEY        optional
  WEB_FALLBACK       1 = try other backends on failure  (default 1)
  WEB_MAX_CHARS      cap on characters returned per call (set by the notebook from the context size)
  WEB_DEADLINE       hard limit in seconds for one tool call (default 80)
Only JSON-RPC goes to stdout; diagnostics go to stderr.
"""
import json, os, re, sys, time, html, threading, urllib.request, urllib.error

BACKEND = os.environ.get("WEB_BACKEND", "parallel").strip().lower() or "parallel"
PARALLEL_KEY = os.environ.get("PARALLEL_API_KEY", "").strip()
EXA_KEY = os.environ.get("EXA_API_KEY", "").strip()
FALLBACK = os.environ.get("WEB_FALLBACK", "1") != "0"
MAX_CHARS = int(os.environ.get("WEB_MAX_CHARS", "12000") or 12000)
DEADLINE = float(os.environ.get("WEB_DEADLINE", "80") or 80)
HTTP_TIMEOUT = 25
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"


def log(*a):
    print("[colab-web]", *a, file=sys.stderr, flush=True)


# ----------------------------------------------------------------- remote MCP client
class RemoteMCP:
    """Minimal Streamable-HTTP MCP client (JSON or SSE replies)."""

    def __init__(self, url, headers=None, timeout=HTTP_TIMEOUT):
        self.url, self.timeout, self.sid = url, timeout, None
        self.headers = {"Content-Type": "application/json",
                        "Accept": "application/json, text/event-stream",
                        "User-Agent": "colab-llama/5"}
        self.headers.update(headers or {})
        self._id = 0

    def _post(self, payload, want_reply=True):
        h = dict(self.headers)
        if self.sid:
            h["Mcp-Session-Id"] = self.sid
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode("utf-8"),
                                     headers=h, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                sid = r.headers.get("Mcp-Session-Id")
                if sid:
                    self.sid = sid
                ctype = (r.headers.get("Content-Type") or "").lower()
                body = r.read().decode("utf-8", errors="replace")   # always UTF-8 (no mojibake)
        except urllib.error.HTTPError as e:
            detail = ""
            try:
                detail = e.read().decode("utf-8", errors="replace")[:500]
            except Exception:
                pass
            raise RuntimeError(f"HTTP {e.code} from {self.url}: {detail}")
        if not want_reply:
            return None
        msgs = []
        if "event-stream" in ctype:
            data_lines = []
            for line in body.splitlines() + [""]:          # SSE: blank line ends an event
                if line.startswith("data:"):
                    data_lines.append(line[5:].lstrip(" "))
                elif line.strip() == "":
                    if data_lines:
                        try:
                            msgs.append(json.loads("\n".join(data_lines)))
                        except ValueError:
                            pass
                        data_lines = []
        elif body.strip():
            j = json.loads(body)
            msgs = j if isinstance(j, list) else [j]
        for m in msgs:
            if isinstance(m, dict) and m.get("id") == payload.get("id"):
                return m
        for m in msgs:                                      # some servers reply with id null on errors
            if isinstance(m, dict) and "error" in m:
                return m
        raise RuntimeError("No reply from MCP server")

    def _rpc(self, method, params=None):
        self._id += 1
        reply = self._post({"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}})
        if "error" in reply:
            err = reply["error"]
            raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
        return reply.get("result", {})

    def call(self, tool, args):
        self._rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                 "clientInfo": {"name": "colab-llama", "version": "5"}})
        try:
            self._post({"jsonrpc": "2.0", "method": "notifications/initialized"}, want_reply=False)
        except Exception:
            pass
        res = self._rpc("tools/call", {"name": tool, "arguments": args})
        text = result_text(res)
        if res.get("isError"):
            raise RuntimeError(text or "tool error")
        return text


def result_text(res):
    parts = []
    for c in res.get("content") or []:
        if isinstance(c, dict) and c.get("type") == "text" and c.get("text"):
            parts.append(c["text"])
    if not parts and res.get("structuredContent") is not None:
        parts.append(json.dumps(res["structuredContent"], ensure_ascii=False, indent=1))
    return "\n".join(parts).strip()


# ----------------------------------------------------------------- backends  (each takes: args, timeout)
def _queries(a):
    q = a.get("query") or ""
    qs = a.get("search_queries") or []
    if isinstance(qs, str):
        qs = [qs]
    qs = [str(x).strip() for x in qs if str(x).strip()]
    if q.strip() and q.strip() not in qs:
        qs.insert(0, q.strip())
    return qs[:5]


def _urls(a):
    u = a.get("urls") or a.get("url") or []
    if isinstance(u, str):
        u = re.split(r"[\s,]+", u)
    out = []
    for x in u:
        x = str(x).strip().strip("<>\"'")
        if x and not re.match(r"^https?://", x, re.I):
            x = "https://" + x
        if x and x not in out:
            out.append(x)
    return out[:10]


def _phdr():
    return {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}


def parallel_search(a, timeout):
    qs = _queries(a)
    objective = (a.get("objective") or "").strip() or (qs[0] if qs else "")
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call(
        "web_search", {"objective": objective[:4000], "search_queries": qs or [objective[:200]]})


def parallel_fetch(a, timeout):
    args = {"urls": _urls(a)}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:200]
    if a.get("full_content"):
        args["full_content"] = True
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_fetch", args)


def parallel_research(a, timeout):
    args = {"input": str(a.get("question") or a.get("input") or a.get("query") or "")[:20000]}
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_research", args)


def _exa(timeout):
    hdr = {"Authorization": f"Bearer {EXA_KEY}"} if EXA_KEY else {}
    return RemoteMCP("https://mcp.exa.ai/mcp", hdr, timeout)


def exa_search(a, timeout):
    qs = _queries(a)
    args = {"query": " ".join(qs[:1]) or a.get("objective", ""), "numResults": 8}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:4000]
    return _exa(timeout).call("web_search_exa", args)


def exa_fetch(a, timeout):
    return _exa(timeout).call("web_fetch_exa", {"urls": _urls(a),
                                                "maxCharacters": 60000 if a.get("full_content") else 8000})


def ddg_search(a, timeout):
    try:
        from ddgs import DDGS
    except ImportError:
        raise RuntimeError("ddgs is not installed (pip install ddgs)")
    out, seen = [], set()
    for q in _queries(a)[:3] or [a.get("objective", "")]:
        try:
            hits = DDGS(timeout=min(15, timeout)).text(q, max_results=8) or []
        except TypeError:                       # very old ddgs without the timeout kwarg
            hits = DDGS().text(q, max_results=8) or []
        for r in hits:
            url = r.get("href") or r.get("url")
            if not url or url in seen:
                continue
            seen.add(url)
            out.append(f"## {r.get('title','')}\nURL: {url}\n{r.get('body','')}")
    if not out:
        raise RuntimeError("DuckDuckGo returned no results")
    return "\n\n".join(out)


def _download(url, timeout):
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html,*/*;q=0.8"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read(3_000_000)
        cs = r.headers.get_content_charset() or "utf-8"
        ctype = (r.headers.get("Content-Type") or "").lower()
    try:
        txt = raw.decode(cs, errors="replace")
    except LookupError:
        txt = raw.decode("utf-8", errors="replace")
    if "html" not in ctype and "<html" not in txt[:2000].lower():
        return txt
    try:
        import trafilatura
        ext = trafilatura.extract(txt, url=url, include_links=False, include_tables=True,
                                  output_format="markdown")
        if ext and len(ext) > 200:
            return ext
    except Exception:
        pass
    txt = re.sub(r"(?is)<(script|style|noscript|svg|head)[^>]*>.*?</\1>", " ", txt)
    txt = re.sub(r"(?s)<[^>]+>", " ", txt)
    return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n\s*\n+", "\n\n", html.unescape(txt))).strip()


def direct_fetch(a, timeout):
    urls, parts = _urls(a), []
    if not urls:
        raise RuntimeError("no valid URL given")
    per = max(3000, MAX_CHARS // len(urls))
    for u in urls[:5]:
        try:
            parts.append(f"# {u}\n{_download(u, min(timeout, 20))[:per]}")
        except Exception as e:
            parts.append(f"# {u}\n(could not load: {e})")
    return "\n\n".join(parts)


SEARCH = {"parallel": parallel_search, "exa": exa_search, "duckduckgo": ddg_search}
FETCH = {"parallel": parallel_fetch, "exa": exa_fetch, "duckduckgo": direct_fetch}


def _call_bounded(fn, args, seconds):
    """Run fn(args, seconds) in a worker thread and give up after `seconds` (the worker is a daemon)."""
    box = {}

    def work():
        try:
            box["r"] = fn(args, seconds)
        except BaseException as e:          # noqa: BLE001 - reported to the caller
            box["e"] = e

    t = threading.Thread(target=work, daemon=True)
    t.start()
    t.join(max(0.5, seconds))
    if t.is_alive():
        raise TimeoutError(f"no answer within {seconds:.0f}s")
    if "e" in box:
        raise box["e"]
    return box["r"]


def run_with_fallback(table, args, budget=None):
    """Try the chosen backend, then the others; the WHOLE call is bounded by `budget` seconds."""
    t0 = time.time()
    budget = DEADLINE if budget is None else budget
    order = [BACKEND] + ([b for b in ("parallel", "duckduckgo", "exa") if b != BACKEND] if FALLBACK else [])
    errors = []
    todo = [b for b in order if b in table]
    for i, b in enumerate(todo):
        left = budget - (time.time() - t0)
        if left < 3:
            errors.append(f"{b}: skipped (out of time)")
            break
        # fair slice: a hung backend must never starve the fallbacks that come after it
        per = min(HTTP_TIMEOUT, max(3.0, (left - 1) / (len(todo) - i)))
        try:
            text = _call_bounded(table[b], args, per)
            if text and text.strip():
                note = "" if b == BACKEND else f"[results from {b}; {BACKEND} was unavailable]\n"
                return note + text
            raise RuntimeError("empty result")
        except Exception as e:              # noqa: BLE001
            msg = str(e) or type(e).__name__
            log(f"{b} failed: {msg[:300]}")
            errors.append(f"{b}: {msg[:160]}")
    raise RuntimeError("all web backends failed - " + " | ".join(errors))


def trim(text, limit):
    if len(text) <= limit:
        return text
    cut = text[:limit]
    nl = cut.rfind("\n")
    if nl > limit * 0.7:
        cut = cut[:nl]
    return cut + "\n\n[...truncated to fit the context window]"


# ----------------------------------------------------------------- MCP server side
TOOLS = [
    {"name": "search",
     "description": ("Search the web for current information. Returns titles, URLs and text excerpts. "
                     "Use for anything recent, factual, or that you are unsure about."),
     "inputSchema": {"type": "object", "properties": {
         "query": {"type": "string", "description": "Short keyword search query, 3-8 words."},
         "objective": {"type": "string", "description": "Optional: one sentence on what you want to find."}},
         "required": ["query"]}},
    {"name": "fetch",
     "description": "Read one or more web pages (e.g. URLs from web_search) and return their main text.",
     "inputSchema": {"type": "object", "properties": {
         "urls": {"type": "array", "items": {"type": "string"}, "description": "Page URLs to read (1-3)."},
         "objective": {"type": "string", "description": "Optional: what to look for on the pages."},
         "full_content": {"type": "boolean", "description": "Return the whole page instead of the relevant parts."}},
         "required": ["urls"]}},
]
if BACKEND == "parallel" and PARALLEL_KEY:
    TOOLS.append({"name": "research",
                  "description": ("Deep research on ONE broad question; returns a cited answer. Slow (up to ~80 s) "
                                  "and uses credits - prefer search + fetch unless the question needs many sources."),
                  "inputSchema": {"type": "object", "properties": {
                      "question": {"type": "string", "description": "The research question."}},
                      "required": ["question"]}})


def handle(msg):
    method, mid = msg.get("method"), msg.get("id")
    if mid is None:
        return None                                          # notification
    if method == "initialize":
        pv = (msg.get("params") or {}).get("protocolVersion") or "2025-06-18"
        return {"protocolVersion": pv, "capabilities": {"tools": {"listChanged": False}},
                "serverInfo": {"name": "colab-web", "version": "5.0"}}
    if method == "ping":
        return {}
    if method == "tools/list":
        return {"tools": TOOLS}
    if method == "tools/call":
        p = msg.get("params") or {}
        name, args = p.get("name"), p.get("arguments") or {}
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                args = {"query": args}
        if not isinstance(args, dict):
            args = {"query": str(args)}
        try:
            if name in ("search", "web_search"):
                if not _queries(args) and not args.get("objective"):
                    raise RuntimeError("web_search needs a 'query'")
                text = run_with_fallback(SEARCH, args)
            elif name in ("fetch", "web_fetch"):
                if not _urls(args):
                    raise RuntimeError("web_fetch needs at least one URL in 'urls'")
                text = run_with_fallback(FETCH, args)
            elif name in ("research", "web_research"):
                if not PARALLEL_KEY:
                    raise RuntimeError("web_research needs a Parallel API key")
                text = _call_bounded(parallel_research, args, max(10, DEADLINE - 3))
            else:
                raise RuntimeError(f"unknown tool {name}")
            return {"content": [{"type": "text", "text": trim(text, MAX_CHARS)}], "isError": False}
        except Exception as e:                  # noqa: BLE001
            note = ("Web tool error: " + (str(e) or type(e).__name__) +
                    "\n(Do not repeat the same call. Either try ONE differently-worded query, or answer now "
                    "from your own knowledge and say that the live lookup failed.)")
            return {"content": [{"type": "text", "text": note}], "isError": True}
    raise KeyError(method)


def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except ValueError:
            continue
        msgs = msg if isinstance(msg, list) else [msg]
        for m in msgs:
            if not isinstance(m, dict):
                continue
            try:
                res = handle(m)
                if res is None:
                    continue
                out = {"jsonrpc": "2.0", "id": m.get("id"), "result": res}
            except KeyError:
                out = {"jsonrpc": "2.0", "id": m.get("id"),
                       "error": {"code": -32601, "message": f"Method not found: {m.get('method')}"}}
            except Exception as e:              # noqa: BLE001
                out = {"jsonrpc": "2.0", "id": m.get("id"), "error": {"code": -32603, "message": str(e)}}
            sys.stdout.write(json.dumps(out, ensure_ascii=False) + "\n")
            sys.stdout.flush()


if __name__ == "__main__":
    main()
'''


def patch_webui(index_path):
    try:
        with open(index_path, encoding="utf-8") as f:
            html = f.read()
        tag = "<script>" + TOGGLE_JS + "</script>\n"
        if "</head>" in html:
            html = html.replace("</head>", tag + "</head>", 1)
        elif "<body" in html:
            html = html.replace("<body", tag + "<body", 1)
        else:
            html = tag + html
        with open(index_path, "w", encoding="utf-8") as f:
            f.write(html)
        return True
    except OSError:
        return False

def ensure_webui(cfg):
    """Copy the official chat UI (prepared by Cell 1, or downloaded now) and add the Allow-tool-calls switch."""
    src = cfg.get("webui_src") or "/content/webui_src"
    try:
        if not os.path.isfile(os.path.join(src, "index.html")):
            tag = cfg.get("ui_tag") or cfg.get("tag")
            print(f"\u23f3 Downloading the chat UI ({tag})...")
            r = requests.get(f"https://github.com/ggml-org/llama.cpp/releases/download/{tag}/llama-{tag}-ui.tar.gz", timeout=180)
            r.raise_for_status()
            with open("/content/webui.tar.gz", "wb") as f:
                f.write(r.content)
            extract_ui("/content/webui.tar.gz", src)
        shutil.rmtree(WEBUI_DIR, ignore_errors=True)
        shutil.copytree(src, WEBUI_DIR)
        if patch_webui(os.path.join(WEBUI_DIR, "index.html")):
            print("\u2705 Chat UI ready (with the Allow tool calls switch)")
        else:
            print("\u26a0\ufe0f Chat UI ready, but the Allow tool calls switch could not be added (the UI asks per tool call instead).")
        return WEBUI_DIR
    except Exception as e:                      # noqa: BLE001
        raise RuntimeError(f"Could not prepare the chat UI ({e}). Check your internet connection and run Cell 2 again.") from e

# ---------------- system prompt (identity + web workflow) ----------------
def build_system_message():
    today = datetime.date.today().strftime("%A, %B %d, %Y")
    parts = [PROFILE["IDENTITY"].format(today=today)]
    if WEB_BACKEND != "off":
        guide = ("WEB TOOLS: you can call web_search (short keyword query, 3-8 words) and web_fetch (read specific URLs). "
                 "Use them for anything recent, factual or uncertain. Workflow: search, then fetch the 1-3 most relevant pages "
                 "if the excerpts are not enough, then WRITE THE ANSWER. Make at most about 6 tool calls for one question, never "
                 "repeat an identical call, and ALWAYS finish with a complete written answer that cites the URLs you used - "
                 "never stop right after a tool result. If a tool returns an error or times out, say so briefly and continue "
                 "with what you know, or try one differently-worded query.")
        if PKEY and WEB_BACKEND == "parallel":
            guide += " web_research is slow and uses credits: only for broad questions that need many sources."
        parts.append(guide)
    return "\n\n".join(parts)

# ---------------- parse the form ----------------
_m = re.match(r"\d+", CONTEXT_WINDOW.strip())
TARGET_CONTEXT = int(_m.group()) if _m else PROFILE["CTX_LADDER"][0]      # "Max ..." -> the top of the ladder
KV_PREF = KV_CACHE.split()[0].lower()
if KV_PREF not in ("auto", "f16", "q8_0", "q4_0"):
    KV_PREF = "auto"
EFFORTS = {"max": "max", "high": "high", "medium": "medium", "low": "low", "minimal": "minimal", "none": "none"}
EFFORT_NUM = {"max": 0.99, "high": 0.9, "medium": 0.7, "low": 0.2, "minimal": 0.1, "none": 0.0}
REASONING_KEY = REASONING.split()[0].lower()
if REASONING_KEY not in EFFORTS:
    REASONING_KEY = "max"
WEB_BACKEND = {"parallel": "parallel", "exa": "exa", "duckduckgo": "duckduckgo"}.get(WEB_SEARCH.split()[0].lower(), "off")
LINK_MODE = PUBLIC_LINK.split()[0].lower()
if LINK_MODE not in ("cloudflare", "off"):
    LINK_MODE = "cloudflare"

if not os.path.exists(CONFIG):
    raise RuntimeError("Run Cell 1 first.")
cfg = json.load(open(CONFIG))
BIN, MODEL_PATH, CF, QUANT = cfg["bin"], cfg["model"], cfg["cloudflared"], cfg["quant"]
for _p in (BIN, MODEL_PATH):
    if not os.path.exists(_p):
        raise RuntimeError(f"Missing {_p}. Run Cell 1 again.")
if LINK_MODE == "cloudflare" and not os.path.exists(CF):
    raise RuntimeError("cloudflared is missing. Run Cell 1 again (or set PUBLIC_LINK to Off).")

PKEY = secret("PARALLEL_API_KEY", PARALLEL_API_KEY) if WEB_BACKEND != "off" else ""
EKEY = secret("EXA_API_KEY", EXA_API_KEY) if WEB_BACKEND != "off" else ""

# ---------------- model facts (from the GGUF itself) ----------------
try:
    META, TENSORS, DATA_START = read_gguf_info(MODEL_PATH)
except Exception as _e:                         # noqa: BLE001
    print(f"\u26a0\ufe0f Could not read the GGUF header ({_e}); using built-in model facts.")
    META, TENSORS, DATA_START = {}, [], 0
N_LAYER, LOOP_START, LOOP_END, LOOP_COUNT, N_KV_HEAD, HEAD_DIM, CTX_TRAIN = arch_facts(META, PROFILE)
FULL_VISITS = full_attn_visits(N_LAYER, LOOP_START, LOOP_END, LOOP_COUNT)
DIMS = (FULL_VISITS, N_KV_HEAD, HEAD_DIM)       # only the global-attention visits hold a long-range KV cache
MODEL_BYTES = os.path.getsize(MODEL_PATH)
MODEL_MIB = MODEL_BYTES / MiB
ALIAS = f"{PROFILE['NAME']}-{QUANT}"

# Which layers may keep their experts in system RAM, and how much GPU memory each one frees. The looped layers never
# qualify: with llama.cpp b126xx, host-resident experts of the looped layers produce garbage output.
HOST_ORDER = host_eligible_layers(N_LAYER, LOOP_START, LOOP_END)
try:
    _eb = expert_bytes_by_layer(TENSORS, DATA_START, MODEL_BYTES)
except Exception:                               # noqa: BLE001
    _eb = {}
_est = 0.9 * MODEL_BYTES / max(1, N_LAYER)      # fallback if the tensor table could not be read
HOST_MIB = [(_eb.get(i) or _est) / MiB for i in HOST_ORDER]
LADDER = PROFILE["CTX_LADDER"]

# ---------------- vision (image input through the mmproj file) ----------------
MMPROJ = cfg.get("mmproj", "")
VISION_STATE = "off"                            # "gpu" -> "cpu" -> "off": degraded automatically if it fails
if globals().get("VISION", False) and MMPROJ and os.path.exists(MMPROJ):
    VISION_STATE = "gpu"
elif globals().get("VISION", False):
    print("\u26a0\ufe0f Vision was requested but the image encoder file is missing - re-run Cell 1 with DOWNLOAD_VISION ticked. Continuing text-only.")

# ---------------- chat template ----------------
def prepare_template():
    emb = META.get("tokenizer.chat_template", "") or ""
    note = "the model's own native template (reasoning effort and tool calls built in)"
    if META and not emb:
        note += " - WARNING: the GGUF carries no chat template, llama.cpp will fall back to ChatML"
    return note

TEMPLATE_NOTE = prepare_template()

# ---------------- web tools (stdio MCP server) ----------------
def write_mcp_config(ctx):
    open(WEB_SCRIPT, "w").write(WEB_MCP_SOURCE)
    env = {"WEB_BACKEND": WEB_BACKEND, "WEB_FALLBACK": "1" if AUTO_FALLBACK else "0",
           "PARALLEL_API_KEY": PKEY, "EXA_API_KEY": EKEY, "PYTHONIOENCODING": "utf-8",
           "WEB_MAX_CHARS": str(tool_cap_chars(ctx)), "WEB_DEADLINE": "80"}
    json.dump({"mcpServers": {"web": {"command": sys.executable, "args": ["-u", WEB_SCRIPT],
                                      "env": env, "timeout_ms": 120000}}}, open(MCP_CONFIG, "w"))
    return env

def probe_web(env):
    print(f"\u23f3 Testing web search ({WEB_BACKEND})... ", end="", flush=True)
    try:
        probe = "\n".join(json.dumps(m) for m in [
            {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "search", "arguments": {"query": "llama.cpp github"}}}]) + "\n"
        out = subprocess.run([sys.executable, "-u", WEB_SCRIPT], input=probe, capture_output=True, text=True,
                             timeout=120, env=dict(os.environ, **env)).stdout
        res = [json.loads(l) for l in out.splitlines() if l.strip().startswith("{")]
        r2 = next((r for r in res if r.get("id") == 2), {}).get("result", {})
        if r2 and not r2.get("isError"):
            txt = r2["content"][0]["text"]
            print("\u2705" + (f"  (note: {txt.splitlines()[0][1:-1]})" if txt.startswith("[results from") else ""))
        else:
            msg = (r2.get("content") or [{"text": "no reply"}])[0]["text"].splitlines()[0]
            print(f"\u26a0\ufe0f {msg[:300]}\n   The tools stay enabled and retry on every call.")
    except Exception as e:                      # noqa: BLE001
        print(f"\u26a0\ufe0f test failed ({e}); the tools stay enabled and retry on every call.")

# ---------------- model server ----------------
OPTIONAL_FLAGS = {"--fit", "--cache-ram", "--sse-ping-interval", "--cors-origins", "--repeat-penalty", "--min-p",
                  "--reasoning-format", "--reasoning-budget", "--reasoning-budget-message", "--ctx-checkpoints",
                  "--ui-config", "--presence-penalty", "--top-k", "--top-p", "--temp", "--image-max-tokens", "--threads-http"}
DROPPED = set()          # optional flags this llama.cpp build rejected (learned at runtime)
SAFE_MODE = [False]      # set when the long-prompt self-test fails: no GPU op-offload, 16-token micro-batches
KV_QUANT_BAD = [False]   # set when this build cannot run a quantized chat memory for this model

class LogPump(threading.Thread):
    """Copies the server's output into LOG and keeps it small: a session of many hours would otherwise write hundreds of MB.
    When LOG passes ~24 MB it is rotated to LOG.1. Never raises: if the disk misbehaves it just discards lines, so the
    server can never block on a full pipe."""
    def __init__(self, proc, path, limit=24 << 20):
        super().__init__(daemon=True)
        self.proc, self.path, self.limit = proc, path, limit
    def run(self):
        f, size = None, 0
        try:
            f = open(self.path, "w", errors="replace")
        except OSError:
            f = None
        for line in self.proc.stdout:
            try:
                if f is not None:
                    f.write(line); f.flush(); size += len(line)
                    if size > self.limit:
                        f.close(); os.replace(self.path, self.path + ".1"); f = open(self.path, "w", errors="replace"); size = 0
            except Exception:                   # noqa: BLE001
                f = None
        try:
            if f is not None:
                f.close()
        except Exception:                       # noqa: BLE001
            pass

def _tail(path, n):
    try:
        with open(path, "rb") as fh:
            fh.seek(0, 2)
            fh.seek(max(0, fh.tell() - n))
            return fh.read().decode("utf-8", "replace")
    except OSError:
        return ""

def exit_reason(proc):
    """Why the process ended, from its exit status ("" while it is still running)."""
    rc = proc.poll() if proc is not None else None
    if rc is None:
        return ""
    if rc < 0:
        names = {9: "killed by the system (almost certainly Colab ran out of RAM)", 11: "crashed (segmentation fault)",
                 6: "aborted (a CUDA / ggml error)", 15: "terminated"}
        return names.get(-rc, f"killed by signal {-rc}")
    return f"exited with code {rc}"

def host_ram():
    """(available MiB, total MiB) of Colab's system RAM."""
    try:
        d = {}
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            d[k] = int(rest.split()[0]) // 1024
        return d.get("MemAvailable"), d.get("MemTotal")
    except Exception:                           # noqa: BLE001
        return None, None

def rss_mib(proc):
    try:
        for line in open(f"/proc/{proc.pid}/status"):
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) // 1024
    except Exception:                           # noqa: BLE001
        pass
    return None

def pkill(name):
    subprocess.run(["pkill", "-x", name], stderr=subprocess.DEVNULL)

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()

def server_command(ctx, kv, k=0, ub=512):
    """llama-server command line. k = number of outer layers whose experts live in system RAM (never a looped layer)."""
    mt = MAX_TOKENS if MAX_TOKENS < 0 else min(MAX_TOKENS, max(256, ctx - 512))
    ui_config = json.dumps({"systemMessage": SYSTEM_MSG, "agenticMaxTurns": 100, "max_tokens": mt,
                            "showThoughtInProgress": bool(SHOW_THOUGHT_LIVE)})
    effort = json.dumps({"reasoning_effort": EFFORTS[REASONING_KEY]})
    cmd = [BIN, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(PORT), "--alias", ALIAS,
           "-ngl", "99", "-c", str(ctx), "--parallel", "1", "-fa", "auto" if kv == "f16" else "on",
           "-ctk", kv, "-ctv", kv, "--jinja", "--chat-template-kwargs", effort]
    def opt(flag, *vals):
        if flag not in DROPPED:
            cmd.extend([flag] + [str(v) for v in vals])
    opt("--fit", "off")                         # the placement is explicit (below); automatic fitting could move looped experts to RAM
    opt("--cache-ram", CACHE_RAM_MIB)
    opt("--threads-http", HTTP_THREADS)
    opt("--ctx-checkpoints", CTX_CHECKPOINTS)
    opt("--sse-ping-interval", 10)
    opt("--cors-origins", "*")                  # MCP tools default CORS to localhost-only; the tunnel/iframe origins differ
    opt("--ui-config", ui_config)
    opt("--temp", TEMP); opt("--top-p", TOP_P); opt("--top-k", TOP_K); opt("--min-p", MIN_P)
    opt("--repeat-penalty", REPEAT_PENALTY)
    if PRESENCE_PENALTY > 0:
        opt("--presence-penalty", PRESENCE_PENALTY)
    opt("--reasoning-format", "deepseek")       # thinking goes to the collapsible panel (reasoning_content)
    if THINKING_BUDGET > 0:
        opt("--reasoning-budget", THINKING_BUDGET)
        opt("--reasoning-budget-message", " Okay, I have thought about this enough. Time to give the final answer.\n")
    if k > 0:
        cmd += ["-ot", host_expert_regex(HOST_ORDER[:k])]
    if k > 0 or SAFE_MODE[0]:
        # llama.cpp b126xx corrupts the prefill of longer prompts when host-resident weights are computed on the GPU
        # ("op offload"), so with experts in RAM the CPU does that work itself. Slower prompt reading, correct answers.
        cmd += ["--no-op-offload"]
    if SAFE_MODE[0]:
        cmd += ["-ub", "16", "-b", "16"]
    else:
        cmd += ["-ub", str(ub)]                  # prompt batch: shrinks for big windows (see compute_mib)
    if VISION_STATE != "off":
        cmd += ["--mmproj", MMPROJ]
        if VISION_STATE == "cpu":
            cmd += ["--no-mmproj-offload"]
        if IMAGE_MAX_TOKENS > 0:
            opt("--image-max-tokens", IMAGE_MAX_TOKENS)
    if WEB_BACKEND != "off":
        cmd += ["--mcp-servers-config", MCP_CONFIG]
    cmd += ["--path", WEBUI_DIR]
    return cmd

def start_server(ctx, kv, k=0, ub=512):
    if WEB_BACKEND != "off":
        write_mcp_config(ctx)
    proc = subprocess.Popen(server_command(ctx, kv, k, ub), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    proc.pump = LogPump(proc, LOG)
    proc.pump.start()
    return proc

def wait_ready(proc, timeout=900):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            if hasattr(proc, "pump"):
                proc.pump.join(timeout=3)           # let the last log lines land before anyone reads them
            return False
        try:
            if LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                time.sleep(1)
                return proc.poll() is None
        except requests.RequestException:
            pass
        time.sleep(2)
    return False

def log_text():
    t = _tail(LOG, 3_000_000)
    return t if len(t) > 200_000 else _tail(LOG + ".1", 1_000_000) + t

def unknown_flag():
    for line in log_text().splitlines():
        l = line.lower()
        if "invalid argument" in l or "error while handling argument" in l or "unknown argument" in l:
            m = re.search(r"(--?[a-z][a-z0-9-]*)", line)
            if m:
                return m.group(1)
    return ""

def fail_reason():
    log = log_text().lower()
    if any(s in log for s in ("graph_reserve", "failed to allocate compute", "compute pp buffers", "compute tg buffers")):
        return "out of memory (working buffers)"        # the attention score matrix of a big window: fix = smaller prompt batch
    if any(s in log for s in ("out of memory", "cudamalloc failed", "failed to allocate", "unable to allocate",
                              "cuda error: out of memory")):
        return "out of memory"
    if "unknown model architecture" in log:
        return "model architecture not supported by this llama.cpp build (Inkling 2 needs b12600 or newer)"
    if "couldn't bind" in log or "address already in use" in log:
        return "port busy"
    if unknown_flag():
        return "bad argument: " + unknown_flag()
    for line in log.splitlines():
        if "template" in line and any(w in line for w in ("error", "failed", "exception", "invalid", "unable")):
            return "chat template error"
    for line in log.splitlines():
        if any(w in line for w in ("mmproj", "clip_", "clip:", "mtmd")) and any(w in line for w in ("error", "failed", "unable", "cannot")):
            return "vision (mmproj) error"
    if "flash" in log and ("not supported" in log or "requires" in log):
        return "flash-attention / KV-quantization problem"
    if "failed to load model" in log or "error loading model" in log:
        return "model failed to load"
    if "no space left on device" in log:
        return "out of memory"
    return "see log"

def fully_on_gpu():
    m = re.search(r"offloaded (\d+)/(\d+) layers to gpu", log_text().lower())
    return m is None or m.group(1) == m.group(2)

def mem_label(kv):
    return {"q8_0": "8-bit chat memory", "q4_0": "4-bit chat memory"}.get(kv, "16-bit chat memory")

def ub_label(ub):
    return "" if ub >= 512 else f"; prompt batch {ub}"

def host_label(k):
    if k <= 0:
        return "all experts on the GPU"
    return f"experts of {k} outer layer{'s' if k != 1 else ''} in system RAM (~{sum(HOST_MIB[:k])/1024:.1f} GiB)"

def try_launch(ctx, kv, k=0, ub=512):
    """Start the server for one (context, kv, host layers) choice, repairing recoverable start-up problems on the way."""
    global PORT, VISION_STATE
    for _ in range(8):
        proc = start_server(ctx, kv, k, ub)
        if wait_ready(proc):
            return proc, ""
        reason = fail_reason()
        stop(proc)
        if reason.startswith("bad argument"):
            flag = reason.split(": ", 1)[1]
            if flag in OPTIONAL_FLAGS and flag not in DROPPED:
                DROPPED.add(flag)
                print(f"\n   (this llama.cpp build does not know {flag} - continuing without it) ", end="", flush=True)
                continue
        if reason == "port busy":
            PORT = free_port(PORT + 1)
            print(f"\n   (port was busy, switching to {PORT}) ", end="", flush=True)
            continue
        if reason == "vision (mmproj) error" and VISION_STATE != "off":
            VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
            print(f"\n   (the image encoder failed - continuing with {'the encoder on the CPU' if VISION_STATE == 'cpu' else 'vision OFF, text only'}) ", end="", flush=True)
            continue
        if reason.startswith("flash-attention") and kv != "f16":
            KV_QUANT_BAD[0] = True
        return None, reason
    return None, "too many start-up retries"

def launch_best(target):
    """launch_best_once(), but if nothing fits with the image encoder on the GPU, move it to the CPU / drop it."""
    global VISION_STATE
    while True:
        try:
            return launch_best_once(target)
        except RuntimeError:
            if VISION_STATE == "off":
                raise
            VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
            print("   Nothing fit with the image encoder on the GPU - trying again with " +
                  ("the encoder on the CPU." if VISION_STATE == "cpu" else "vision OFF (text only)."))

def launch_best_once(target):
    used0, total = gpu_mem()
    # Subtract at most 1 GiB of "already used" memory: right after a crash the old process may still show up in
    # nvidia-smi for a moment. If something really does hog the GPU, the load fails and the next step is tried.
    avail = (total - min(used0 or 0, 1024)) if total else 15000
    overhead = PROFILE["OVERHEAD_MIB"] + (PROFILE.get("VISION_MIB", 0) if VISION_STATE == "gpu" else 0)
    kmax = len(HOST_MIB)

    def plan():
        pref = "f16" if (KV_QUANT_BAD[0] and KV_PREF != "f16") else KV_PREF
        c = plan_candidates(MODEL_MIB, avail, LADDER, target, pref, DIMS, overhead, MIN_FREE_MIB,
                            auto_order=PROFILE.get("KV_AUTO_ORDER"), host_mib=HOST_MIB, force_k=HOST_EXPERT_LAYERS, ub_user=UBATCH,
                            max_extra_k=2 if CONTEXT_WINDOW.strip().lower().startswith("max") else None)
        return c or [(LADDER[-1], "f16" if KV_QUANT_BAD[0] else "q4_0", kmax, UB_LADDER[-1])]

    k_floor = min(HOST_EXPERT_LAYERS, kmax) if HOST_EXPERT_LAYERS >= 0 else kmax
    floor = (MODEL_MIB - sum(HOST_MIB[:k_floor]) + overhead + MIN_FREE_MIB + kv_mib(LADDER[-1], "q4_0", *DIMS)
             + compute_mib(LADDER[-1], UB_LADDER[-1]))
    if total and floor > avail:
        raise RuntimeError(
            f"This model file ({MODEL_MIB/1024:.1f} GiB) is too large for the {avail/1024:.1f} GiB GPU even with the experts of every outer layer "
            f"in system RAM. In Cell 1 pick a smaller quant (IQ3_S or IQ2_M), run Cell 1 again, then this cell.")
    queue = plan()
    first = queue[0]
    print(f"   plan: model {MODEL_MIB/1024:.1f} GiB + chat memory + working buffers ({compute_mib(first[0], first[3])/1024:.1f} GiB) + {overhead} MiB overhead"
          f"{' (incl. the image encoder)' if VISION_STATE == 'gpu' else ''} on a {avail/1024:.1f} GiB GPU  ->  {host_label(first[2])}{ub_label(first[3])}")
    n_tried, fatal = 0, False
    while queue and n_tried < 24 and not fatal:
        ctx, kv, k, ub = queue.pop(0)
        if kv != "f16" and KV_QUANT_BAD[0]:
            continue
        while n_tried < 24:
            n_tried += 1
            print(f"   trying {ctx:>9,} tokens ({mem_label(kv)}; {host_label(k)}{ub_label(ub)}) ... ", end="", flush=True)
            proc, reason = try_launch(ctx, kv, k, ub)
            if proc is not None and not fully_on_gpu():
                stop(proc); proc, reason = None, "model did not fully fit on the GPU"
            if proc is not None:
                used, tot = gpu_mem()
                if used is not None and tot - used < MIN_FREE_MIB:
                    stop(proc); proc, reason = None, f"fits, but only {tot-used} MiB GPU memory left (unsafe for long chats)"
            if proc is not None:
                print("\u2705")
                return proc, ctx, kv, k, ub
            print(f"\u274c ({reason})")
            if reason.startswith("bad argument") or reason in ("model failed to load", "chat template error",
                    "vision (mmproj) error") or reason.startswith("model architecture not supported"):
                fatal = True
                break
            if reason.startswith("flash-attention") and kv != "f16":
                print("   (this build cannot use a quantized chat memory for this model - using 16-bit)")
                queue = plan()
                break
            if reason == "out of memory (working buffers)":
                # more experts in RAM would not help: the attention score matrix is too big for this window. Halve the prompt batch.
                if UBATCH <= 0 and not SAFE_MODE[0] and ub > UB_LADDER[-1]:
                    ub //= 2
                    continue
                break
            tight = reason == "out of memory" or reason == "model did not fully fit on the GPU" or reason.startswith("fits, but only")
            if tight and HOST_EXPERT_LAYERS < 0 and k < kmax:
                k = min(kmax, k + 2)            # the estimate was too optimistic: move two more outer layers' experts to RAM, same window
                continue
            break
        time.sleep(3)
    print(log_text()[-3000:])
    raise RuntimeError("Could not start the model - see the log above.")

# ---------------- vision self-test ----------------
def _msg_text(resp):
    m = resp["choices"][0]["message"]
    return ((m.get("content") or "") + " " + (m.get("reasoning_content") or "")).strip()

def vision_test():
    """Send a tiny solid-red PNG and ask for its colour. -> (server_ok, answer_text_or_error)."""
    import base64
    b64 = base64.b64encode(make_png_solid(128, 128, (220, 20, 20))).decode()
    try:
        r = LOCAL.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", timeout=600, json={
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": "data:image/png;base64," + b64}},
                {"type": "text", "text": "What is the main colour of this image? Answer with one word."}]}],
            "max_tokens": 500, "temperature": 0.2, "chat_template_kwargs": {"reasoning_effort": "none"}})
        if r.status_code != 200:
            return False, f"HTTP {r.status_code}: {r.text[:160]}"
        return True, _msg_text(r.json())[-200:]
    except Exception as e:                      # noqa: BLE001
        return False, str(e)[:160]

def prefill_test():
    """A ~300-token prompt. Some llama.cpp b126xx memory layouts corrupt the prefill of prompts longer than ~96 tokens
    (the reply is then all '?' and the server stays broken). -> (ok, note)."""
    filler = " ".join(f"item{i}" for i in range(1, 121))
    try:
        r = LOCAL.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", timeout=600, json={
            "messages": [{"role": "user", "content": "Here is a list: " + filler +
                          "\n\nIgnore the list. What is the capital of France? Answer in one word."}],
            "max_tokens": 500, "temperature": 0.2, "chat_template_kwargs": {"reasoning_effort": "none"}})
        if r.status_code != 200:
            return False, f"HTTP {r.status_code}: {r.text[:160]}"
        text = _msg_text(r.json())
    except Exception as e:                      # noqa: BLE001
        return False, str(e)[:160]
    if not text.strip():
        return False, "empty answer after a long prompt"
    if text.count("?") > 0.5 * len(text):
        return False, "garbled answer (all '?') after a long prompt"
    return True, "" if "paris" in text.lower() else "unexpected answer, but readable"

def http_threads():
    """Number of web-server threads llama-server reports at start-up (None if this build does not print it)."""
    m = re.search(r"http threads:\s*(\d+)", log_text())
    return int(m.group(1)) if m else None

def concurrency_test():
    """llama-server gives every open connection one thread of a small pool (3 on a 2-core Colab). Streaming replies, queued
    requests and stale browser connections can occupy all of them; then the chat page itself ("new chat" -> /props) hangs and
    the public link answers 502 / Error 500 until they clear. Open 6 chat requests, then ask for the page endpoint 8 times.
    -> (ok, note)"""
    base = f"http://127.0.0.1:{PORT}"
    stop_evt, streams = threading.Event(), []

    def hold():
        r = None
        try:
            r = LOCAL.post(base + "/v1/chat/completions", stream=True, timeout=60, json={
                "messages": [{"role": "user", "content": "Count from 1 to 200, one number per line."}],
                "max_tokens": 400, "stream": True, "temperature": 0.2,
                "chat_template_kwargs": {"reasoning_effort": "none"}})
            streams.append(r)
            for _ in r.iter_lines():
                if stop_evt.is_set():
                    break
        except Exception:                       # noqa: BLE001
            pass
        finally:
            try:
                if r is not None:
                    r.close()
            except Exception:                   # noqa: BLE001
                pass

    holders = [threading.Thread(target=hold, daemon=True) for _ in range(6)]
    for t in holders:
        t.start()
    time.sleep(1.5)
    answered = []

    def probe():
        try:
            if LOCAL.get(base + "/props", timeout=10).status_code == 200:
                answered.append(1)
        except Exception:                       # noqa: BLE001
            pass

    t0 = time.time()
    probes = [threading.Thread(target=probe, daemon=True) for _ in range(8)]
    for t in probes:
        t.start()
    for t in probes:
        t.join(timeout=15)
    took = time.time() - t0
    stop_evt.set()
    for r in list(streams):
        try:
            r.close()
        except Exception:                       # noqa: BLE001
            pass
    for t in holders:
        t.join(timeout=5)
    time.sleep(3)                               # let the server cancel the abandoned requests
    n = len(answered)
    return n == 8, f"{n}/8 page requests answered in {took:.1f} s while 6 chat requests were open"

# ---------------- self tests (non-fatal) ----------------
def smoke_test(ctx):
    base = f"http://127.0.0.1:{PORT}"
    try:
        p = LOCAL.get(base + "/props", timeout=10).json()
        mods = p.get("modalities") or {}
        if mods:
            print("   input types the server accepts: text" + "".join(f", {k}" for k, v in mods.items() if v))
        n_ctx = (p.get("default_generation_settings") or {}).get("n_ctx")
        if n_ctx:
            print(f"   server reports a {n_ctx:,}-token window" + ("" if n_ctx >= ctx else f" (asked for {ctx:,})"))
    except Exception as e:                      # noqa: BLE001
        print(f"   (could not read /props: {str(e)[:80]})")
    if WEB_BACKEND != "off":
        try:
            names = [t.get("tool") or t.get("name") for t in LOCAL.get(base + "/tools", timeout=15).json()]
            web = [n for n in names if n and n.startswith("web_")]
            print(("   \u2705 web tools exposed to the model: " + ", ".join(sorted(web))) if web else
                  "   \u26a0\ufe0f the server lists no web_* tools - see /content/llama-server.log")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f could not list tools: {str(e)[:100]}")
    try:
        t0 = time.time()
        r = LOCAL.post(base + "/v1/chat/completions", timeout=300, json={
            "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
            "max_tokens": 300, "temperature": 0.2, "chat_template_kwargs": {"reasoning_effort": "none"}})
        r.raise_for_status()
        j = r.json()
        u = j.get("usage", {})
        has_panel = bool((j["choices"][0]["message"].get("reasoning_content") or "").strip())
        print(f"   \u2705 generation works ({u.get('completion_tokens', '?')} tokens in {time.time()-t0:.1f} s"
              f"{'; thinking is split into its own panel' if has_panel else ''})")
    except Exception as e:                      # noqa: BLE001
        print(f"   \u26a0\ufe0f test generation failed: {str(e)[:160]}")
        return
    if WEB_BACKEND != "off":
        try:
            tools = [{"type": "function", "function": {"name": "web_search", "description": "search the web",
                      "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
            r = LOCAL.post(base + "/v1/chat/completions", timeout=300, json={
                "messages": [{"role": "user", "content": "Find the newest llama.cpp release."}],
                "tools": tools, "tool_choice": "required", "max_tokens": 600, "temperature": 0.2,
                "chat_template_kwargs": {"reasoning_effort": "none"}})
            r.raise_for_status()
            calls = (r.json()["choices"][0]["message"].get("tool_calls") or [])
            print("   \u2705 tool-call parsing works" if calls else
                  "   \u26a0\ufe0f the model answered without a parsed tool call in the self-test (usually still fine in chat)")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f tool-call self-test failed: {str(e)[:200]}")

# ---------------- public link ----------------
def tunnel_ready():
    try:
        return LOCAL.get(f"http://127.0.0.1:{METRICS_PORT}/ready", timeout=3).status_code == 200
    except requests.RequestException:
        return False

def url_answers(url):
    try:
        return requests.get(url + "/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

def start_cloudflare():
    pkill("cloudflared")
    time.sleep(1)
    for attempt in range(3):
        p = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                              "--protocol", "http2", "--no-autoupdate",
                              "--metrics", f"127.0.0.1:{METRICS_PORT}"],
                             stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(60):
            if p.poll() is not None:
                break
            try:
                found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
                         if not u.startswith("https://api.")]
            except OSError:
                found = []
            if found:
                url = found[0]
                break
            time.sleep(1)
        if url:
            for _ in range(30):   # wait until the link actually answers
                if p.poll() is not None:
                    break
                if tunnel_ready() and url_answers(url):
                    return p, url
                time.sleep(2)
            if p.poll() is None and tunnel_ready():
                return p, url          # connected; the edge may just need a few more seconds
        stop(p)
        time.sleep(3)
    return None, None

def link_alive(handle):
    return handle is not None and handle.poll() is None and tunnel_ready()

def show_link(url, title):
    display(HTML(f'<div style="font-size:16px;margin:8px 0">{title} <b><a href="{url}" target="_blank">{url}</a></b></div>'))

def server_healthy():
    try:
        return LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

# ================= main =================
def main():
    global PORT, METRICS_PORT, SYSTEM_MSG, VISION_STATE
    pkill("llama-server")
    pkill("cloudflared")
    time.sleep(2)
    PORT, METRICS_PORT = free_port(8090), free_port(20241)
    SYSTEM_MSG = build_system_message()
    ensure_webui(cfg)
    web_env = write_mcp_config(TARGET_CONTEXT) if WEB_BACKEND != "off" else {}
    if WEB_BACKEND != "off":
        probe_web(web_env)

    print(f"\u23f3 Loading {PROFILE['NAME']} {QUANT} - target: " +
          ("the largest context that fits" if CONTEXT_WINDOW.strip().lower().startswith("max") else f"{TARGET_CONTEXT:,} tokens") + "...")
    print(f"   chat template: {TEMPLATE_NOTE}")
    print(f"   architecture: {N_LAYER} stored layers, layers {LOOP_START}-{LOOP_END - 1} looped x{LOOP_COUNT} ({N_LAYER + (LOOP_END - LOOP_START) * (LOOP_COUNT - 1)} layer visits per token); "
          f"{FULL_VISITS} global-attention visits keep the long-range chat memory")
    server, CTX, KV, KH, UB = launch_best(TARGET_CONTEXT)

    def banner():
        used, total = gpu_mem()
        think_txt = (f"{REASONING_KEY.upper()} (effort {EFFORT_NUM[REASONING_KEY]}, " +
                     ("unlimited thinking" if THINKING_BUDGET <= 0 else f"up to {THINKING_BUDGET:,} thinking tokens") + ")")
        print(f"\n\u2705 Model ready  |  model: {QUANT}  |  context: {CTX:,} tokens ({mem_label(KV)})  |  reasoning: {think_txt}")
        print(f"   {host_label(KH)}  |  GPU memory: {used} / {total} MiB")
        if UB < 512 and not SAFE_MODE[0]:
            print(f"   \u2139\ufe0f Prompt batch is {UB} tokens (a big window needs small working buffers): long prompts and files are read more slowly.")
        if "Max" not in CONTEXT_WINDOW and CTX < TARGET_CONTEXT:
            print(f"   \u2139\ufe0f {TARGET_CONTEXT:,} didn't fit in GPU memory; using {CTX:,}.")
        if CTX <= 8192 and REASONING_KEY in ("max", "high"):
            print(f"   \u2139\ufe0f {CTX:,} tokens is small for {REASONING_KEY.upper()} reasoning (thinking can use thousands of tokens, and this model cannot shift its context): "
                  "if a chat stops with \u201ccontext full\u201d, pick 16384 or more in the form and run Cell 2 again.")
        web_txt = "OFF"
        if WEB_BACKEND != "off":
            web_txt = "ON \u00b7 " + WEB_BACKEND + (" + auto-fallback" if AUTO_FALLBACK else "")
        cap_txt = f"  |  one web result is capped at {tool_cap_chars(CTX):,} characters" if WEB_BACKEND != "off" else ""
        print(f"   web search: {web_txt}{cap_txt}")
    banner()
    while VISION_STATE != "off":
        print(f"\u23f3 Testing image input (vision on the {'GPU' if VISION_STATE == 'gpu' else 'CPU'})... ", end="", flush=True)
        ok, ans = vision_test()
        if ok:
            print(f"\u2705  the model sees images (test answer: \u201c{ans[-40:]}\u201d)" +
                  ("" if "red" in ans.lower() else "  (unexpected answer - check with a real image)"))
            break
        print(f"\u26a0\ufe0f failed ({ans})")
        stop(server)
        VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
        print("   Restarting with " + ("the encoder on the CPU (slower image encoding)..." if VISION_STATE == "cpu" else "vision OFF (text only)..."))
        server, CTX, KV, KH, UB = launch_best(TARGET_CONTEXT)
    if globals().get("VISION", False) and VISION_STATE == "off":
        print("   \u2139\ufe0f Running text-only: image attachments will not work this session.")

    print("\u23f3 Testing a long prompt... ", end="", flush=True)
    ok, why = prefill_test()
    if not ok and not SAFE_MODE[0]:
        print(f"\u26a0\ufe0f {why}")
        print("   Known llama.cpp issue with this model (ggml-org/llama.cpp #32778). Restarting in safe mode "
              "(no GPU op-offload, 16-token micro-batches - slower prompt reading, but correct)...")
        SAFE_MODE[0] = True
        stop(server)
        server, CTX, KV, KH, UB = launch_best(TARGET_CONTEXT)
        print("\u23f3 Testing a long prompt again... ", end="", flush=True)
        ok, why = prefill_test()
    print("\u2705 long prompts work" + (f"  ({why})" if why else "") + ("  [safe mode]" if SAFE_MODE[0] else "") if ok else
          f"\u26a0\ufe0f {why} - long prompts may fail; see Troubleshooting. Short chats still work.")
    smoke_test(CTX)
    nt = http_threads()
    print("\u23f3 Testing that the chat page stays reachable while the model is busy... ", end="", flush=True)
    ok, note = concurrency_test()
    if ok:
        print(f"\u2705 {note}" + (f"  (web-server threads: {nt})" if nt else ""))
    else:
        print(f"\u26a0\ufe0f {note}" + (f"  (web-server threads: {nt})" if nt else ""))
        print("   Opening a new chat while the model is answering may show 502 / Error 500 for a moment. "
              "Raise HTTP_THREADS (advanced block) and run Cell 2 again; if it persists, your llama.cpp build may ignore --threads-http.")
    print()

    tunnel, public_url = None, None
    try:
        from google.colab import output
        # Colab's private new-tab links (*.prod.colab.dev) return 404 in modern browsers,
        # so only the in-notebook view is shown; use the public link for a full tab.
        if SHOW_INLINE:
            print("\U0001F4AC Chat inside the notebook (private, only you):")
            output.serve_kernel_port_as_iframe(PORT, height=850)
        if LINK_MODE != "off":
            print(f"\u23f3 Opening the public link ({LINK_MODE})...")
            tunnel, public_url = start_cloudflare()
            if public_url:
                show_link(public_url, "\U0001F30D Public link:")
                print("   (If it says it can't be reached, wait 30 s and refresh, or open it in an incognito window.)")
            else:
                print("\u26a0\ufe0f The public link didn't start yet - it keeps retrying below. The chat above works meanwhile.")

        print("\nHow to use: open a link \u2192 type your message. Long text/code (1,000+ lines) is fine if the context window is big enough: paste it or attach the file.")
        if WEB_BACKEND != "off":
            print("Web search: use the \u201cAllow tool calls: ON / OFF\u201d switch inside the chat box, right after the + button, to control whether searches ask for permission (applies instantly, no reload).")
        print("Keep this cell running. Press \u25a0 (stop) to shut everything down.\n")

        start, last_report = time.time(), 0
        server_restarts = tunnel_restarts = bad_ticks = bad_health = 0
        restart_times, crash_files = [], 0
        while True:
            time.sleep(20)

            # 1) model server: restart on crash (smaller context if it ran out of memory) or if it stops answering.
            #    While it restarts, the chat shows "502" / "Error 500": wait for the "Model server is back" line, then refresh.
            crashed = server.poll() is not None
            if not crashed:
                bad_health = 0 if server_healthy() else bad_health + 1
            hung = bad_health >= 9                   # ~3 minutes without a single /health answer
            if crashed or hung:
                now = time.time()
                restart_times = [t for t in restart_times if now - t < 1800]
                if len(restart_times) >= MAX_SERVER_RESTARTS:
                    print(f"\u274c Model server failed {MAX_SERVER_RESTARTS} times in 30 minutes - giving up. Last log lines:\n")
                    print(log_text()[-3000:])
                    break
                restart_times.append(now)
                server_restarts += 1
                why = exit_reason(server)
                reason = "hung (no answer for ~3 min)" if hung else fail_reason()
                ram_free, ram_total = host_ram()
                # keep the evidence: the log is overwritten by the next start, so save this one
                crash_files += 1
                crash_path = f"/content/llama-server.crash-{crash_files}.log"
                try:
                    open(crash_path, "w").write(log_text()[-400000:])
                except OSError:
                    crash_path = ""
                stop(server)
                target = CTX
                if reason.startswith("out of memory"):
                    smaller = [c for c in LADDER if c < CTX]
                    if smaller:
                        target = smaller[0]
                        print(f"\u26a0\ufe0f Ran out of GPU memory at {CTX:,} tokens - restarting with at most {target:,}.")
                print(f"\u26a0\ufe0f Model server stopped - {why or 'it stopped answering'}; log says: {reason}. "
                      f"Colab RAM free: {ram_free if ram_free is not None else '?'} / {ram_total if ram_total is not None else '?'} MiB.")
                tail = [l for l in log_text().splitlines()[-14:] if l.strip()]
                if tail:
                    print("   last log lines:\n     " + "\n     ".join(l[:200] for l in tail) +
                          (f"\n   (saved to {crash_path})" if crash_path else ""))
                print(f"   restarting ({len(restart_times)}/{MAX_SERVER_RESTARTS} in the last 30 min) - the chat shows 502 until it is back...")
                try:
                    server, CTX, KV, KH, UB = launch_best(target)
                    bad_health = 0
                    print("\u2705 Model server is back. Refresh the chat page if it still shows an error; your chats are kept in your browser, just resend the last message.")
                except RuntimeError as e:
                    print(f"\u274c {e}")
                continue

            # 2) public link
            if LINK_MODE != "off":
                bad_ticks = 0 if link_alive(tunnel) else bad_ticks + 1
                if tunnel is None or bad_ticks >= 3:
                    tunnel_restarts += 1
                    print(f"\u26a0\ufe0f Public link dropped - reconnecting (#{tunnel_restarts})...")
                    stop(tunnel)
                    tunnel, new_url = start_cloudflare()
                    bad_ticks = 0
                    if new_url and new_url != public_url:
                        public_url = new_url
                        show_link(public_url, "\U0001F30D NEW public link (the old one no longer works):")
                    elif new_url:
                        print("\u2705 Public link reconnected (same address).")

            # 3) heartbeat every 10 min
            mins = int((time.time() - start) // 60)
            if mins - last_report >= 10:
                last_report = mins
                ram_free, ram_total = host_ram()
                used, total = gpu_mem()
                rss = rss_mib(server)
                print(f"\u23f1 Up {mins} min | server restarts: {server_restarts} | link restarts: {tunnel_restarts} | "
                      f"RAM free {ram_free if ram_free is not None else '?'}/{ram_total if ram_total is not None else '?'} MiB"
                      f"{f' (server uses {rss} MiB)' if rss else ''} | GPU {used}/{total} MiB")
                if ram_free is not None and ram_free < 700:
                    print("\u26a0\ufe0f Colab RAM is almost full - the system may kill the model server soon. "
                          "Start a fresh chat, or use a smaller quant / HOST_EXPERT_LAYERS = 0 next time.")
    except KeyboardInterrupt:
        print("Stopping...")
    finally:
        stop(tunnel)
        stop(server)
        print("Shut down.")

if __name__ == "__main__":
    main()


## Troubleshooting

- **“model architecture not supported by this llama.cpp build”** → the pinned llama.cpp must be **b12600 or newer** (`LLAMA_TAG` in Cell 1). Use a newer tag only if it still ships `inkling2_loop`; delete `/content/llama.cpp` and the Drive cache file, then run Cell 1.
- **Slow generation on a T4 with Q4_K** → expected: some experts run on the 2 Colab CPU cores. Pick **IQ3_S** (fewer layers in RAM) or **IQ2_M** (all on the GPU) in Cell 1, or an L4 / A100 runtime. `HOST_EXPERT_LAYERS` in the advanced block overrides the automatic choice (0 = never use RAM; it then fails to load if the file does not fit).
- **“Server error (502)” or “Error 500 – Internal Error” when you open a new chat while the model is answering, fixed by reloading a few times** → the model is fine; the chat page could not get a web-server thread. llama.cpp gives each open connection one thread of a small pool (3 on a 2-core Colab), and streaming replies, queued requests and the browser's idle connections can take them all, so the page's own requests hang and the tunnel answers 502. Cell 2 now starts the server with 32 threads (`HTTP_THREADS`) and tests it at start-up (“Testing that the chat page stays reachable while the model is busy…”). If that line shows ⚠️, raise `HTTP_THREADS` and run Cell 2 again.
- **“Model server stopped – …” appears in Cell 2's output** → that is a real restart (the chat shows 502 until “Model server is back”, then refresh). The message names the cause (e.g. “killed by the system – Colab ran out of RAM”), prints the last log lines and saves the whole log as `/content/llama-server.crash-N.log`. Up to 5 restarts within 30 minutes are allowed. Every 10 minutes the heartbeat line shows free RAM and GPU memory.
- **The chat gets slow or glitchy after hours of Max-reasoning work** → huge thinking texts are heavy for the browser. The reasoning panel now stays collapsed while it streams (`SHOW_THOUGHT_LIVE = False`; click it to read). Start a fresh chat now and then, open the public link in its own browser tab instead of the view inside the notebook, and remember that a big window means a small prompt batch (slow prompt reading).
- **“out of memory (working buffers)” / “failed to allocate compute pp buffers”** → the window is too big for the attention score matrix at the current prompt batch. Cell 2 halves the batch (down to 16) and then falls back to a smaller window by itself; moving experts to RAM cannot help with this one. To avoid the loop, pick a smaller window in the form.
- **Long prompts are read slowly with a big window** → expected: big windows need a tiny prompt batch. Pick a smaller window (e.g. 32768) to get a larger batch.
- **“context full” / the answer stops in the middle of thinking** → the default 4,096-token window is small for Max reasoning. Pick **16384** (or more) in Cell 2 and run it again, or lower the reasoning level.
- **The model thinks for a very long time / loops** → that is Max reasoning on a small model. Pick **High** or **Medium**, set `THINKING_BUDGET = 8192` (best effort), lower `MAX_TOKENS`, or press stop. If it repeats itself, try `PRESENCE_PENALTY = 0.5`.
- **Output is all “?” or garbage, and stays broken** → the known prefill bug of llama.cpp b126xx (#32778). Cell 2 tests for it and restarts in safe mode by itself; if you still see it, set `UBATCH = 16` and `HOST_EXPERT_LAYERS = 0` (only if the file fits), or try a newer `LLAMA_TAG`.
- **“the image encoder failed” / “encoder on the CPU”** → Cell 2 handled it for you. CPU encoding works but takes tens of seconds per image. A smaller quant or a smaller context frees GPU memory; then re-run Cell 2.
- **Images are ignored** → check that Cell 1 printed “Vision: mmproj-…” (tick *DOWNLOAD_VISION* and re-run Cell 1 if not) and that Cell 2 printed “the model sees images”.
- **Chat memory shows 16-bit although I picked 8/4-bit** → this llama.cpp build could not run a quantized chat memory for this model (flash-attention limitation), so Cell 2 fell back to 16-bit.
- **Context is lower than you picked** → that is the most your GPU can hold; Cell 2 tells you what it used.
- **Chat stops after a web search / long answer** → web results are capped to a fraction of the context window, every web call is bounded to 80 s, and the chat UI may take up to 100 agent steps. If it still happens, open `/content/llama-server.log` (the last lines say why) and use the **Allow tool calls** switch to see whether a tool is being blocked.
- **Web search says rate-limited / failed** → it falls back automatically (Parallel → DuckDuckGo → Exa). For heavy use, add a free key from platform.parallel.ai as a secret named `PARALLEL_API_KEY`.
- **Tool prompts keep appearing / never appear** → use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button. The choice is saved in your browser. If the switch is missing, hard-refresh once (Ctrl+Shift+R / Cmd+Shift+R).
- **“this llama.cpp build does not know --xyz”** → harmless: Cell 2 drops optional flags a build does not understand and carries on.
- **Public link doesn't open** → wait ~30 s and refresh. Cloudflare quick tunnels take a moment to propagate. The link changes every run; the monitoring loop reconnects it and prints the new address if it drops.
- **“q8_0 / 4-bit chat memory”** → that is the conversation cache (KV cache), not the model. The model file is shown as `model:`.
- **Colab disconnects** → free Colab stops idle or long sessions (usually after a few hours). Re-run Cell 1 (fast: everything finished is reused, and the Drive cache skips the compile) and then Cell 2.
- **Download or build failed** → just run Cell 1 again; finished parts are reused. The model is 16.5 GB: make sure the runtime has ~32 GB of free disk.
- **Logs** → `/content/build.log`, `/content/llama-server.log`, `/content/tunnel.log`.
